# Torre de Londres · Busca em largura

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesLB/dcc-014-trabalho-1/blob/main/apps/notebook/notebooks/03_busca_em_largura.ipynb)

DCC014 Inteligência Artificial · UFJF · Trabalho 1 · Grupo 5

Páginas: [Projeto: problema, regras e motor de busca](https://colab.research.google.com/github/CharlesLB/dcc-014-trabalho-1/blob/main/apps/notebook/notebooks/00_projeto.ipynb) · [Busca irrevogável](https://colab.research.google.com/github/CharlesLB/dcc-014-trabalho-1/blob/main/apps/notebook/notebooks/01_busca_irrevogavel.ipynb) · [Backtracking](https://colab.research.google.com/github/CharlesLB/dcc-014-trabalho-1/blob/main/apps/notebook/notebooks/02_backtracking.ipynb) · **Busca em largura** · [Busca ordenada](https://colab.research.google.com/github/CharlesLB/dcc-014-trabalho-1/blob/main/apps/notebook/notebooks/04_busca_ordenada.ipynb) · [Gráficos e análise](https://colab.research.google.com/github/CharlesLB/dcc-014-trabalho-1/blob/main/apps/notebook/notebooks/05_analise.ipynb)

Não escolhe regra: aplica todas, os filhos esperam numa fila e a árvore é varrida por níveis.

Nada de profundidade d+1 é visitado antes de esgotar a profundidade d, então o primeiro caminho até um estado é o mais curto até ele. O objetivo encontrado é o ótimo em número de movimentos.

## Preparação

Rode a célula a seguir antes de qualquer outra. Ela grava todo o `src/` do repositório, menos `src/core/algorithms/breadth_first.py`, que aparece por inteiro logo abaixo, nos mesmos caminhos, e põe `src/` no `sys.path`. A página roda sozinha: não depende de outra página nem de clonar o repositório. Para ver o código de cada pasta, abra a página [Projeto: problema, regras e motor de busca](https://colab.research.google.com/github/CharlesLB/dcc-014-trabalho-1/blob/main/apps/notebook/notebooks/00_projeto.ipynb).

In [ ]:
#@title Grava o código do projeto
import sys
from pathlib import Path

FILES = {
    'src/config/logging.py': 'from __future__ import annotations\n\nimport logging\nimport sys\nfrom typing import Final\n\nLOGGER_NAME: Final = "torre_de_londres"\nLOG_FORMAT: Final = "%(levelname)s %(name)s: %(message)s"\n\n\ndef configure_logging(*, verbose: bool = False) -> logging.Logger:\n    logger = logging.getLogger(LOGGER_NAME)\n    logger.setLevel(logging.DEBUG if verbose else logging.WARNING)\n    if not logger.handlers:\n        handler = logging.StreamHandler(sys.stderr)\n        handler.setFormatter(logging.Formatter(LOG_FORMAT))\n        logger.addHandler(handler)\n    return logger\n\n\ndef get_logger(suffix: str) -> logging.Logger:\n    return logging.getLogger(f"{LOGGER_NAME}.{suffix}")\n',
    'src/config/settings.py': 'from __future__ import annotations\n\nfrom collections.abc import Mapping\nfrom typing import Final\n\nMAX_ITERATIONS: Final = 10_000\n\nDEFAULT_OUTPUT_FORMAT: Final = "console"\nDEFAULT_RANK_MODE: Final = "lexicographic"\n\nOUTPUT_FORMATS: Final[tuple[str, ...]] = ("console", "json")\nRANK_MODES: Final[tuple[str, ...]] = ("lexicographic", "score")\n\nEXIT_OK: Final = 0\nEXIT_INVALID_INPUT: Final = 2\n\n\nPROGRAM_NAME: Final = "main.py"\n\nDESCRIPTION: Final = (\n    "Resolvedor da Torre de Londres: executa busca irrevogável, backtracking, "\n    "busca em largura e busca ordenada sobre as cartas do problema, variando a "\n    "estratégia de controle, e ranqueia os resultados."\n)\n\nEPILOG: Final = (\n    \'Cada eixo omitido significa "todos": sem --problem roda todas as cartas, \'\n    "sem --algorithm roda todos os algoritmos, sem --strategy roda todas as "\n    "estratégias."\n)\n\nHELP_HELP: Final = "Mostra esta ajuda e encerra."\nHELP_PROBLEM: Final = "Cartas a resolver (repetível ou separado por vírgula)."\nHELP_ALGORITHM: Final = "Algoritmos a executar (repetível ou separado por vírgula)."\nHELP_STRATEGY: Final = "Estratégias de controle (repetível ou separado por vírgula)."\nHELP_ALL: Final = (\n    "Produto cartesiano completo: todas as cartas, algoritmos e estratégias."\n)\nHELP_ALL_GOALS: Final = (\n    "Resolve todos os 36 estados finais possíveis a partir da posição inicial "\n    "e mostra só o resumo, com média e mediana de cada algoritmo e estratégia."\n)\nHELP_ORDER: Final = "Ordem das regras para a estratégia custom, ex.: R2,R4,R6,R1,R3,R5."\nHELP_SHOW_TREE: Final = "Renderiza a árvore de busca de cada execução."\nHELP_SHOW_TRACE: Final = (\n    "Mostra o passo a passo com regra aplicada e estado resultante."\n)\nHELP_SHOW_STATES: Final = "Desenha as hastes em cada estado do caminho solução."\nHELP_FORMAT: Final = "Formato da saída."\nHELP_OUTPUT: Final = "Arquivo de destino da saída; sem ele, escreve no terminal."\nHELP_SVG_DIR: Final = (\n    "Pasta onde gravar a árvore de busca de cada execução em DOT e SVG "\n    "(o SVG exige o Graphviz instalado)."\n)\nHELP_RANK_MODE: Final = "Critério do placar."\nHELP_MAX_ITERATIONS: Final = "Guarda contra execução patológica."\nHELP_SEED: Final = "Reservado; a execução é determinística por construção."\nHELP_LIST: Final = "Lista cartas, algoritmos e estratégias disponíveis e encerra."\nHELP_VERBOSE: Final = "Habilita log de diagnóstico na saída de erro."\n\nKIND_LABELS: Final[Mapping[str, str]] = {\n    "problem": "Carta não reconhecida",\n    "algorithm": "Algoritmo não reconhecido",\n    "strategy": "Estratégia não reconhecida",\n    "format": "Formato de saída não reconhecido",\n    "rank_mode": "Modo de ranqueamento não reconhecido",\n}\n\nERROR_PREFIX: Final = "Erro:"\nERROR_UNKNOWN_SELECTION: Final = "{kind}: {name}. Disponíveis: {available}."\nERROR_INVALID_ARGUMENT: Final = "argumento inválido: {detail}."\nERROR_INVALID_RULE_ORDER: Final = "ordem de regras inválida: {reason}."\nERROR_HINT: Final = "Use --help para ver as opções."\n\nUSAGE_PREFIX: Final = "Uso: "\nOPTIONS_TITLE: Final = "Opções"\n\nLIST_PROBLEMS_HEADER: Final = "Cartas disponíveis"\nLIST_ALGORITHMS_HEADER: Final = "Algoritmos disponíveis"\nLIST_STRATEGIES_HEADER: Final = "Estratégias disponíveis"\nOUTPUT_WRITTEN: Final = "Saída gravada em {path}"\nGRAPHS_WRITTEN: Final = "{count} arquivos de árvore gravados em {path}"\nGRAPHVIZ_MISSING: Final = (\n    "Aviso: Graphviz (dot) não encontrado; só os arquivos .dot foram gravados."\n)\n',
    'src/core/rules/domain/base.py': '"""Legenda do problema.\n\nDiscos: VERDE, VERMELHO, AZUL. Distintos por cor; NÃO há ordenação por tamanho.\nHastes: H1, H2, H3.\n\nCapacidades:\n    H1: até 3 discos\n    H2: até 2 discos\n    H3: até 1 disco\n\nA capacidade é a única restrição estrutural: é o que substitui a ordenação por\ntamanho da Torre de Hanói.\n\nEstado: tupla de três pilhas, cada uma da base para o topo. Imutável, hashable e\ncanônica: uma configuração física tem exatamente uma escrita.\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Mapping\nfrom enum import IntEnum, unique\nfrom types import MappingProxyType\nfrom typing import Protocol\n\n\n@unique\nclass Disk(IntEnum):\n    GREEN = 0\n    RED = 1\n    BLUE = 2\n\n\n@unique\nclass Peg(IntEnum):\n    H1 = 0\n    H2 = 1\n    H3 = 2\n\n\nCAPACITIES: Mapping[Peg, int] = MappingProxyType({Peg.H1: 3, Peg.H2: 2, Peg.H3: 1})\n\nTOTAL_DISKS: int = len(Disk)\n\ntype Stack = tuple[Disk, ...]\ntype State = tuple[Stack, Stack, Stack]\n\n\nclass TransitionRule(Protocol):\n    """Um movimento: tira o disco do topo de uma haste e põe no topo de outra.\n\n        is_applicable(estado)  ->  posso mover agora? (origem tem disco e\n                                   destino tem espaço)\n        apply(estado)          ->  o estado que resulta do movimento\n        cost                   ->  quanto custa mover (só a busca ordenada usa)\n\n    `apply` não mexe no estado recebido, devolve um novo -- por isso um nó da\n    árvore nunca perde a configuração que guardava. São seis regras fixas, R1\n    a R6, uma por par origem/destino, definidas em `moves.py`.\n\n    A busca não sabe o que é disco nem haste: ela só testa a regra, aplica a\n    que passou, e segue. Trocar o problema é trocar as regras, não o motor.\n    """\n\n    @property\n    def id(self) -> str: ...\n\n    @property\n    def origin(self) -> Peg: ...\n\n    @property\n    def destination(self) -> Peg: ...\n\n    @property\n    def cost(self) -> int: ...\n\n    def is_applicable(self, state: State) -> bool: ...\n\n    def apply(self, state: State) -> State: ...\n',
    'src/core/rules/domain/catalog.py': 'from __future__ import annotations\n\nfrom collections.abc import Mapping\nfrom types import MappingProxyType\n\nfrom core.rules.domain.base import TransitionRule\nfrom core.rules.domain.exceptions import UnknownRuleError\nfrom core.rules.moves import R1, R2, R3, R4, R5, R6\n\nRULES: tuple[TransitionRule, ...] = (R1, R2, R3, R4, R5, R6)\n\nRULE_BY_ID: Mapping[str, TransitionRule] = MappingProxyType(\n    {rule.id: rule for rule in RULES}\n)\n\nINVERSE_RULE_ID: Mapping[str, str] = MappingProxyType(\n    {"R1": "R3", "R3": "R1", "R2": "R5", "R5": "R2", "R4": "R6", "R6": "R4"}\n)\n\n\ndef get_rule(rule_id: str) -> TransitionRule:\n    try:\n        return RULE_BY_ID[rule_id]\n    except KeyError:\n        raise UnknownRuleError(rule_id) from None\n\n\ndef inverse_of(rule: TransitionRule) -> TransitionRule:\n    return RULE_BY_ID[INVERSE_RULE_ID[rule.id]]\n',
    'src/core/rules/domain/constraints.py': '"""Condições de validade.\n\n    1. Origem: a haste de origem possui pelo menos um disco no momento exato da\n       ação.\n    2. Destino: a haste de destino não ultrapassa sua capacidade estrita de\n       volume (H1 até 3, H2 até 2, H3 até 1).\n\nAs duas são estáticas: dependem apenas do estado corrente.\n\nA terceira condição do material, que proíbe produzir estado já presente no\ncaminho, é contextual: depende do percurso, e por isso vive no algoritmo\n(`core/search_tree/path.py`), não aqui.\n"""\n\nfrom __future__ import annotations\n\nfrom core.rules.domain.base import CAPACITIES, Peg, State\n\n\ndef has_disk_to_move(state: State, origin: Peg) -> bool:\n    return len(state[origin]) >= 1\n\n\ndef has_room_for_disk(state: State, destination: Peg) -> bool:\n    return len(state[destination]) + 1 <= CAPACITIES[destination]\n\n\ndef is_move_allowed(state: State, origin: Peg, destination: Peg) -> bool:\n    return has_disk_to_move(state, origin) and has_room_for_disk(state, destination)\n',
    'src/core/rules/domain/exceptions.py': 'class RuleError(Exception): ...\n\n\nclass RuleNotApplicableError(RuleError):\n    def __init__(self, rule_id: str) -> None:\n        super().__init__(f"rule {rule_id} is not applicable to the given state")\n        self.rule_id = rule_id\n\n\nclass UnknownRuleError(RuleError):\n    def __init__(self, rule_id: str) -> None:\n        super().__init__(f"unknown rule identifier: {rule_id!r}")\n        self.rule_id = rule_id\n\n\nclass InvalidRuleOrderError(RuleError):\n    def __init__(self, reason: str) -> None:\n        super().__init__(f"invalid rule order: {reason}")\n        self.reason = reason\n\n\nclass UnknownStrategyError(RuleError):\n    def __init__(self, name: str) -> None:\n        super().__init__(f"unknown control strategy: {name!r}")\n        self.name = name\n',
    'src/core/rules/moves.py': '"""Regras de transição de estado.\n\n    R1: mover o disco do topo da H1 para a H2.\n    R2: mover o disco do topo da H1 para a H3.\n    R3: mover o disco do topo da H2 para a H1.\n    R4: mover o disco do topo da H2 para a H3.\n    R5: mover o disco do topo da H3 para a H1.\n    R6: mover o disco do topo da H3 para a H2.\n\nToda regra tem inversa exata: R1<->R3, R2<->R5, R4<->R6.\n\nCusto de uma regra: a distância entre as hastes que ela liga, como um braço\nque leva o disco de uma haste à outra.\n\n    R1, R3 (H1 <-> H2): 1     R4, R6 (H2 <-> H3): 1     R2, R5 (H1 <-> H3): 2\n\nUma regra e sua inversa custam o mesmo. Só a busca ordenada olha o custo.\n\n`apply` exige `is_applicable` como pré-condição: aplicar regra inválida é erro de\nprogramação e falha alto.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nfrom core.rules.domain.base import Peg, State\nfrom core.rules.domain.constraints import is_move_allowed\nfrom core.rules.domain.exceptions import RuleNotApplicableError\n\n\n@dataclass(frozen=True, slots=True)\nclass Move:\n    id: str\n    origin: Peg\n    destination: Peg\n\n    @property\n    def cost(self) -> int:\n        return abs(self.destination - self.origin)\n\n    def is_applicable(self, state: State) -> bool:\n        return is_move_allowed(state, self.origin, self.destination)\n\n    def apply(self, state: State) -> State:\n        if not self.is_applicable(state):\n            raise RuleNotApplicableError(self.id)\n\n        stacks = list(state)\n        disk = stacks[self.origin][-1]\n        stacks[self.origin] = stacks[self.origin][:-1]\n        stacks[self.destination] = (*stacks[self.destination], disk)\n        return (stacks[0], stacks[1], stacks[2])\n\n\nR1 = Move("R1", Peg.H1, Peg.H2)\nR2 = Move("R2", Peg.H1, Peg.H3)\nR3 = Move("R3", Peg.H2, Peg.H1)\nR4 = Move("R4", Peg.H2, Peg.H3)\nR5 = Move("R5", Peg.H3, Peg.H1)\nR6 = Move("R6", Peg.H3, Peg.H2)\n',
    'src/core/rules/strategies/domain/base.py': 'from __future__ import annotations\n\nfrom typing import Protocol\n\nfrom core.rules.domain.base import TransitionRule\n\n\nclass ControlStrategy(Protocol):\n    @property\n    def name(self) -> str: ...\n\n    def order(\n        self, rules: tuple[TransitionRule, ...]\n    ) -> tuple[TransitionRule, ...]: ...\n',
    'src/core/rules/strategies/domain/registry.py': 'from __future__ import annotations\n\nfrom collections.abc import Mapping\nfrom types import MappingProxyType\n\nfrom core.rules.domain.exceptions import UnknownStrategyError\nfrom core.rules.strategies.ascending import ASCENDING\nfrom core.rules.strategies.custom_order import CUSTOM\nfrom core.rules.strategies.descending import DESCENDING\nfrom core.rules.strategies.domain.base import ControlStrategy\n\nSTRATEGIES: Mapping[str, ControlStrategy] = MappingProxyType(\n    {\n        ASCENDING.name: ASCENDING,\n        DESCENDING.name: DESCENDING,\n        CUSTOM.name: CUSTOM,\n    }\n)\n\nSTRATEGY_NAMES: tuple[str, ...] = tuple(STRATEGIES)\n\n\ndef get_strategy(name: str) -> ControlStrategy:\n    try:\n        return STRATEGIES[name]\n    except KeyError:\n        raise UnknownStrategyError(name) from None\n',
    'src/core/rules/strategies/ascending.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\n\nfrom core.rules.domain.base import TransitionRule\nfrom core.rules.domain.catalog import RULES\n\n_CANONICAL_POSITION = {rule.id: position for position, rule in enumerate(RULES)}\n\n\n@dataclass(frozen=True, slots=True)\nclass AscendingStrategy:\n    name: str = "ascending"\n\n    def order(self, rules: tuple[TransitionRule, ...]) -> tuple[TransitionRule, ...]:\n        return tuple(sorted(rules, key=lambda rule: _CANONICAL_POSITION[rule.id]))\n\n\nASCENDING = AscendingStrategy()\n',
    'src/core/rules/strategies/custom_order.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\nfrom core.rules.domain.base import TransitionRule\nfrom core.rules.domain.catalog import RULE_BY_ID, RULES\nfrom core.rules.domain.exceptions import InvalidRuleOrderError\n\nDEFAULT_CUSTOM_ORDER: tuple[str, ...] = ("R2", "R4", "R6", "R1", "R3", "R5")\n\n\n@dataclass(frozen=True, slots=True)\nclass CustomOrderStrategy:\n    sequence: tuple[str, ...] = DEFAULT_CUSTOM_ORDER\n    name: str = field(default="custom")\n\n    def __post_init__(self) -> None:\n        known = set(RULE_BY_ID)\n        given = set(self.sequence)\n        if len(self.sequence) != len(given):\n            raise InvalidRuleOrderError("duplicated rule identifiers")\n        unknown = given - known\n        if unknown:\n            raise InvalidRuleOrderError(f"unknown rule identifiers: {sorted(unknown)}")\n        missing = known - given\n        if missing:\n            raise InvalidRuleOrderError(f"missing rule identifiers: {sorted(missing)}")\n\n    def order(self, rules: tuple[TransitionRule, ...]) -> tuple[TransitionRule, ...]:\n        position = {rule_id: index for index, rule_id in enumerate(self.sequence)}\n        return tuple(sorted(rules, key=lambda rule: position[rule.id]))\n\n\ndef parse_order(raw: str) -> tuple[str, ...]:\n    identifiers = tuple(\n        token.strip().upper() for token in raw.split(",") if token.strip()\n    )\n    if len(identifiers) != len(RULES):\n        raise InvalidRuleOrderError(\n            f"expected {len(RULES)} identifiers, got {len(identifiers)}"\n        )\n    return CustomOrderStrategy(sequence=identifiers).sequence\n\n\nCUSTOM = CustomOrderStrategy()\n',
    'src/core/rules/strategies/descending.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\n\nfrom core.rules.domain.base import TransitionRule\nfrom core.rules.domain.catalog import RULES\n\n_CANONICAL_POSITION = {rule.id: position for position, rule in enumerate(RULES)}\n\n\n@dataclass(frozen=True, slots=True)\nclass DescendingStrategy:\n    name: str = "descending"\n\n    def order(self, rules: tuple[TransitionRule, ...]) -> tuple[TransitionRule, ...]:\n        return tuple(\n            sorted(rules, key=lambda rule: _CANONICAL_POSITION[rule.id], reverse=True)\n        )\n\n\nDESCENDING = DescendingStrategy()\n',
    'src/core/domain/disk.py': 'from core.rules.domain.base import TOTAL_DISKS, Disk\n\n__all__ = ["TOTAL_DISKS", "Disk"]\n',
    'src/core/domain/peg.py': 'from core.rules.domain.base import CAPACITIES, Peg\n\n__all__ = ["CAPACITIES", "Peg"]\n',
    'src/core/domain/problem.py': '"""Estado inicial e catálogo de cartas.\n\nInicial, fixo pelo aparato:  H1[VERDE, VERMELHO]  H2[AZUL]  H3[]\nCada carta define um objetivo a alcançar a partir dele.\n\n`Problem` valida inicial e objetivo na construção: carta impossível falha na\nimportação do módulo, não no meio de uma busca. O campo `initial` tem padrão para\nque testes montem cenários sintéticos sem tocar no catálogo.\n\n`all_goal_problems` monta uma carta sintética por estado do espaço (G01 a\nG36), todas a partir do mesmo inicial: é a bateria usada para comparar os\nalgoritmos por média e mediana em vez de por uma carta só.\n\nÚnico ponto a alterar quando o conjunto de cartas muda: os testes usam\npropriedades relativas, nunca o literal do objetivo.\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Mapping\nfrom dataclasses import dataclass\nfrom types import MappingProxyType\n\nfrom core.domain.state import State, build_state, validate\nfrom core.domain.state_space import all_states\nfrom core.rules.domain.base import Disk\n\nGREEN = Disk.GREEN\nRED = Disk.RED\nBLUE = Disk.BLUE\n\n\nclass UnknownProblemError(Exception):\n    def __init__(self, problem_id: str) -> None:\n        super().__init__(f"unknown problem identifier: {problem_id!r}")\n        self.problem_id = problem_id\n\n\nINITIAL_STATE: State = build_state((GREEN, RED), (BLUE,), ())\n\n\n@dataclass(frozen=True, slots=True)\nclass Problem:\n    id: str\n    goal: State\n    initial: State = INITIAL_STATE\n\n    def __post_init__(self) -> None:\n        validate(self.goal)\n        validate(self.initial)\n\n    def is_goal(self, state: State) -> bool:\n        return state == self.goal\n\n\nPROBLEMS: tuple[Problem, ...] = (Problem("P1", build_state((), (RED, GREEN), (BLUE,))),)\n\nPROBLEM_BY_ID: Mapping[str, Problem] = MappingProxyType(\n    {problem.id: problem for problem in PROBLEMS}\n)\n\nPROBLEM_IDS: tuple[str, ...] = tuple(PROBLEM_BY_ID)\n\n\ndef get_problem(problem_id: str) -> Problem:\n    try:\n        return PROBLEM_BY_ID[problem_id]\n    except KeyError:\n        raise UnknownProblemError(problem_id) from None\n\n\ndef all_goal_problems(initial: State = INITIAL_STATE) -> tuple[Problem, ...]:\n    return tuple(\n        Problem(f"G{index:02d}", goal, initial=initial)\n        for index, goal in enumerate(all_states(), start=1)\n    )\n',
    'src/core/domain/state.py': 'from __future__ import annotations\n\nfrom collections.abc import Iterable\n\nfrom core.rules.domain.base import CAPACITIES, TOTAL_DISKS, Disk, Peg, Stack, State\n\n__all__ = [\n    "CAPACITIES",\n    "EMPTY_STATE",\n    "TOTAL_DISKS",\n    "Disk",\n    "InvalidStateError",\n    "Peg",\n    "Stack",\n    "State",\n    "build_state",\n    "disks_of",\n    "is_valid",\n    "stack_of",\n    "top_of",\n    "validate",\n]\n\nEMPTY_STATE: State = ((), (), ())\n\n\nclass InvalidStateError(Exception):\n    def __init__(self, reason: str) -> None:\n        super().__init__(f"invalid state: {reason}")\n        self.reason = reason\n\n\ndef build_state(h1: Iterable[Disk], h2: Iterable[Disk], h3: Iterable[Disk]) -> State:\n    state: State = (tuple(h1), tuple(h2), tuple(h3))\n    validate(state)\n    return state\n\n\ndef validate(state: State) -> None:\n    if len(state) != len(Peg):\n        raise InvalidStateError(f"expected {len(Peg)} pegs, got {len(state)}")\n\n    for peg in Peg:\n        capacity = CAPACITIES[peg]\n        if len(state[peg]) > capacity:\n            raise InvalidStateError(\n                f"peg {peg.name} holds {len(state[peg])} disks, capacity is {capacity}"\n            )\n\n    disks = disks_of(state)\n    if len(disks) != TOTAL_DISKS:\n        raise InvalidStateError(\n            f"expected {TOTAL_DISKS} distinct disks, got {len(disks)}"\n        )\n\n\ndef is_valid(state: State) -> bool:\n    try:\n        validate(state)\n    except InvalidStateError:\n        return False\n    return True\n\n\ndef disks_of(state: State) -> frozenset[Disk]:\n    return frozenset(disk for stack in state for disk in stack)\n\n\ndef stack_of(state: State, peg: Peg) -> Stack:\n    return state[peg]\n\n\ndef top_of(state: State, peg: Peg) -> Disk | None:\n    stack = state[peg]\n    return stack[-1] if stack else None\n',
    'src/core/domain/state_space.py': '"""Enumeração exaustiva do espaço de estados.\n\nNão participa de busca alguma: descreve o espaço por força bruta e serve de\noráculo independente aos testes.\n\nContagem:\n    3! = 6 permutações x 10 cortes em três partes = 60 disposições\n    as capacidades (3, 2, 1) deixam 6 distribuições:\n        (3,0,0)  (2,1,0)  (2,0,1)  (1,2,0)  (1,1,1)  (0,2,1)\n    6 distribuições x 6 arranjos = 36 estados válidos\n\nPropriedades:\n    - Grafo não dirigido: toda regra tem inversa exata.\n    - Grafo conexo: todo objetivo é alcançável de qualquer inicial. Logo\n      backtracking e largura sempre terminam em SUCESSO neste problema; só a\n      busca irrevogável pode terminar em IMPASSE.\n    - Fator de ramificação: 2 a 4 regras aplicáveis por estado.\n    - Profundidade máxima de um caminho sem repetição: 36.\n\n`shortest_distance` é uma busca em largura escrita à parte do motor, sem árvore,\nnó ou estratégia. É o oráculo contra o qual a otimalidade de `breadth_first` é\nverificada. `cheapest_cost` faz o mesmo papel para a busca ordenada: um\nDijkstra sobre o custo das regras, também fora do motor.\n"""\n\nfrom __future__ import annotations\n\nimport heapq\nfrom collections import deque\nfrom collections.abc import Iterator\nfrom itertools import permutations\n\nfrom core.domain.state import Disk, State, is_valid\nfrom core.rules.domain.base import TransitionRule\nfrom core.rules.domain.catalog import RULES\n\n\ndef all_states() -> tuple[State, ...]:\n    return tuple(_generate_states())\n\n\ndef _generate_states() -> Iterator[State]:\n    disks = tuple(Disk)\n    for arrangement in permutations(disks):\n        for first in range(len(disks) + 1):\n            for second in range(first, len(disks) + 1):\n                candidate: State = (\n                    arrangement[:first],\n                    arrangement[first:second],\n                    arrangement[second:],\n                )\n                if is_valid(candidate):\n                    yield candidate\n\n\ndef applicable_rules(state: State) -> tuple[TransitionRule, ...]:\n    return tuple(rule for rule in RULES if rule.is_applicable(state))\n\n\ndef successors(state: State) -> tuple[tuple[TransitionRule, State], ...]:\n    return tuple((rule, rule.apply(state)) for rule in applicable_rules(state))\n\n\ndef reachable_from(source: State) -> frozenset[State]:\n    seen = {source}\n    queue = deque([source])\n    while queue:\n        current = queue.popleft()\n        for _, successor in successors(current):\n            if successor not in seen:\n                seen.add(successor)\n                queue.append(successor)\n    return frozenset(seen)\n\n\ndef shortest_distance(source: State, target: State) -> int | None:\n    if source == target:\n        return 0\n    depths = {source: 0}\n    queue = deque([source])\n    while queue:\n        current = queue.popleft()\n        for _, successor in successors(current):\n            if successor in depths:\n                continue\n            depths[successor] = depths[current] + 1\n            if successor == target:\n                return depths[successor]\n            queue.append(successor)\n    return None\n\n\ndef cheapest_cost(source: State, target: State) -> int | None:\n    costs = {source: 0}\n    heap: list[tuple[int, int, State]] = [(0, 0, source)]\n    pushed = 1\n    while heap:\n        cost, _, current = heapq.heappop(heap)\n        if current == target:\n            return cost\n        if cost > costs[current]:\n            continue\n        for rule, successor in successors(current):\n            candidate = cost + rule.cost\n            if candidate < costs.get(successor, candidate + 1):\n                costs[successor] = candidate\n                heapq.heappush(heap, (candidate, pushed, successor))\n                pushed += 1\n    return None\n',
    'src/core/search_tree/frontier.py': '"""FRONTEIRA — a lista de ABERTOS.\n\nOs nós já gerados que ainda não foram processados. A busca tira um daqui,\nolha em volta e devolve os filhos. Fronteira vazia, busca encerrada.\n\nQuem sai primeiro define o algoritmo: pilha afunda, fila varre por nível,\nfila de prioridade sai pelo menor custo.\n"""\n\nfrom __future__ import annotations\n\nimport heapq\nfrom collections import deque\nfrom typing import Protocol\n\nfrom core.search_tree.node import Node\n\n\nclass EmptyFrontierError(Exception):\n    """Tentou tirar um nó de uma fronteira vazia."""\n\n    def __init__(self) -> None:\n        super().__init__("cannot pop from an empty frontier")\n\n\nclass Frontier(Protocol):\n    """Guardar um nó, tirar um nó, contar quantos faltam. Só isso.\n\n    Sem índice, sem busca por estado, sem percorrer o conteúdo: o único\n    acesso é pela ponta. O contrato não diz QUAL nó o pop devolve — é a\n    única coisa que muda entre as implementações, e é o que define a busca.\n    """\n\n    def push(self, node: Node) -> None:\n        """Guarda um nó. Ele sempre entra pelo fim."""\n        ...\n\n    def pop(self) -> Node:\n        """Tira um nó e devolve. QUAL nó é o que muda entre as fronteiras."""\n        ...\n\n    def __len__(self) -> int:\n        """Quantos nós ainda esperam. Zero encerra a busca."""\n        ...\n\n\nclass StackFrontier:\n    """PILHA (LIFO): sai o último que entrou. Usada pelo backtracking.\n\n    ```\n    push S1, S2, S3  ->  [S1 S2 S3]  ->  pop devolve S3\n    ```\n\n    Entra e sai pelo mesmo lado, então o filho recém-gerado é sempre o\n    próximo a ser olhado: a busca afunda num ramo até o fim.\n    """\n\n    __slots__ = ("_nodes",)\n\n    def __init__(self) -> None:\n        self._nodes: list[Node] = []\n\n    def push(self, node: Node) -> None:\n        """Empilha o nó no topo."""\n        self._nodes.append(node)\n\n    def pop(self) -> Node:\n        """Tira e devolve o nó do topo: o ÚLTIMO que entrou."""\n        if not self._nodes:\n            raise EmptyFrontierError\n        return self._nodes.pop()\n\n    def __len__(self) -> int:\n        """Quantos nós estão empilhados."""\n        return len(self._nodes)\n\n\nclass QueueFrontier:\n    """FILA (FIFO): sai o primeiro que entrou. Usada pela busca em largura.\n\n    ```\n    push S1, S2, S3  ->  [S1 S2 S3]  ->  pop devolve S1\n    ```\n\n    Entra por um lado, sai pelo outro, então os filhos esperam atrás de todo\n    o nível atual: a busca varre nível por nível e acha o caminho mais curto.\n\n    Usa deque, não list, porque popleft é O(1) e list.pop(0) é O(n).\n    """\n\n    __slots__ = ("_nodes",)\n\n    def __init__(self) -> None:\n        self._nodes: deque[Node] = deque()\n\n    def push(self, node: Node) -> None:\n        """Enfileira o nó no fim da fila."""\n        self._nodes.append(node)\n\n    def pop(self) -> Node:\n        """Tira e devolve o nó do início: o PRIMEIRO que entrou.\n\n        É a única linha que difere da StackFrontier, e é ela que troca\n        profundidade por largura.\n        """\n        if not self._nodes:\n            raise EmptyFrontierError\n        return self._nodes.popleft()\n\n    def __len__(self) -> int:\n        """Quantos nós estão na fila."""\n        return len(self._nodes)\n\n\nclass PriorityFrontier:\n    """FILA ORDENADA POR CUSTO: sai o de menor custo acumulado. Usada pela\n    busca ordenada.\n\n    ```\n    push S1(5), S2(6), S3(6), S4(1)  ->  pop devolve S4, S1, S2, S3\n    ```\n\n    Empate de custo sai o gerado primeiro (menor `order`). Irmãos de mesmo\n    custo nascem na ordem da estratégia, então ela é o desempate entre eles.\n\n    Heap de (custo, ordem, nó): a ordem é única, então o nó nunca é comparado.\n    """\n\n    __slots__ = ("_heap", "_removed")\n\n    def __init__(self) -> None:\n        self._heap: list[tuple[int, int, Node]] = []\n        self._removed: set[int] = set()\n\n    def push(self, node: Node) -> None:\n        """Insere o nó na posição do seu custo."""\n        heapq.heappush(self._heap, (node.cost, node.order, node))\n\n    def pop(self) -> Node:\n        """Tira e devolve o nó de MENOR custo; no empate, o mais antigo."""\n        while self._heap:\n            _, order, node = heapq.heappop(self._heap)\n            if order in self._removed:\n                self._removed.discard(order)\n                continue\n            return node\n        raise EmptyFrontierError\n\n    def remove(self, node: Node) -> None:\n        """Tira um nó que ainda está na fila, sem devolvê-lo.\n\n        A remoção é preguiçosa: o nó só é descartado quando chegaria a vez\n        dele no pop. O nó precisa estar na fila.\n        """\n        self._removed.add(node.order)\n\n    def __len__(self) -> int:\n        """Quantos nós ainda esperam, sem contar os removidos."""\n        return len(self._heap) - len(self._removed)\n',
    'src/core/search_tree/metrics.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\n\n@dataclass(frozen=True, slots=True)\nclass SearchMetrics:\n    iterations: int\n    nodes_generated: int\n    nodes_visited: int\n    rules_tested: int\n    backtracks: int\n    deadlocks: int\n    max_depth: int\n    elapsed_ms: float\n\n\n@dataclass(slots=True)\nclass MetricsCollector:\n    iterations: int = 0\n    nodes_generated: int = 0\n    nodes_visited: int = 0\n    rules_tested: int = 0\n    backtracks: int = 0\n    deadlocks: int = 0\n    max_depth: int = field(default=0)\n\n    def count_iteration(self) -> None:\n        self.iterations += 1\n\n    def count_generated(self, depth: int) -> None:\n        self.nodes_generated += 1\n        self.max_depth = max(self.max_depth, depth)\n\n    def count_visited(self) -> None:\n        self.nodes_visited += 1\n\n    def count_rule_test(self) -> None:\n        self.rules_tested += 1\n\n    def count_backtrack(self) -> None:\n        self.backtracks += 1\n\n    def count_deadlock(self) -> None:\n        self.deadlocks += 1\n\n    def seal(self, elapsed_ms: float) -> SearchMetrics:\n        return SearchMetrics(\n            iterations=self.iterations,\n            nodes_generated=self.nodes_generated,\n            nodes_visited=self.nodes_visited,\n            rules_tested=self.rules_tested,\n            backtracks=self.backtracks,\n            deadlocks=self.deadlocks,\n            max_depth=self.max_depth,\n            elapsed_ms=elapsed_ms,\n        )\n',
    'src/core/search_tree/node.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\n\nfrom core.domain.state import State\nfrom core.rules.domain.base import TransitionRule\n\n\n@dataclass(frozen=True, slots=True)\nclass Node:\n    state: State\n    parent: Node | None\n    rule: TransitionRule | None\n    depth: int\n    order: int\n    cost: int = 0\n\n    @property\n    def is_root(self) -> bool:\n        return self.parent is None\n\n    def __hash__(self) -> int:\n        return hash((self.order, self.state))\n',
    'src/core/search_tree/outcome.py': 'from __future__ import annotations\n\nfrom enum import Enum, unique\n\n\n@unique\nclass Outcome(Enum):\n    SUCCESS = "SUCCESS"\n    DEADLOCK = "DEADLOCK"\n    CUTOFF = "CUTOFF"\n    FAILURE = "FAILURE"\n\n    @property\n    def is_success(self) -> bool:\n        return self is Outcome.SUCCESS\n',
    'src/core/search_tree/path.py': 'from __future__ import annotations\n\nfrom collections.abc import Iterator\n\nfrom core.domain.state import State\nfrom core.search_tree.node import Node\n\n\ndef ancestors(node: Node) -> Iterator[Node]:\n    current = node.parent\n    while current is not None:\n        yield current\n        current = current.parent\n\n\ndef path_to(node: Node) -> tuple[Node, ...]:\n    reversed_path = [node, *ancestors(node)]\n    reversed_path.reverse()\n    return tuple(reversed_path)\n\n\ndef states_on_path(node: Node) -> frozenset[State]:\n    return frozenset(step.state for step in path_to(node))\n\n\ndef contains_state(node: Node, state: State) -> bool:\n    if node.state == state:\n        return True\n    return any(ancestor.state == state for ancestor in ancestors(node))\n\n\ndef applied_rules(node: Node) -> tuple[str, ...]:\n    return tuple(step.rule.id for step in path_to(node) if step.rule is not None)\n',
    'src/core/search_tree/result.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\n\nfrom core.search_tree.metrics import SearchMetrics\nfrom core.search_tree.node import Node\nfrom core.search_tree.outcome import Outcome\nfrom core.search_tree.trace import Trace\n\n\n@dataclass(frozen=True, slots=True)\nclass SearchResult:\n    algorithm: str\n    strategy: str\n    problem: str\n    outcome: Outcome\n    solution_path: tuple[Node, ...]\n    applied_rules: tuple[str, ...]\n    metrics: SearchMetrics\n    trace: Trace\n\n    @property\n    def solution_length(self) -> int | None:\n        return len(self.applied_rules) if self.outcome.is_success else None\n\n    @property\n    def solution_cost(self) -> int | None:\n        return self.solution_path[-1].cost if self.outcome.is_success else None\n\n    @property\n    def label(self) -> str:\n        return f"{self.algorithm} · {self.strategy}"\n',
    'src/core/search_tree/trace.py': 'from __future__ import annotations\n\nfrom collections.abc import Iterator\nfrom dataclasses import dataclass, field\nfrom enum import Enum, unique\n\nfrom core.domain.state import State\nfrom core.search_tree.node import Node\n\n\n@unique\nclass TraceEvent(Enum):\n    ROOT = "ROOT"\n    VISIT = "VISIT"\n    GENERATE = "GENERATE"\n    PRUNE = "PRUNE"\n    GOAL = "GOAL"\n    DEADLOCK = "DEADLOCK"\n    BACKTRACK = "BACKTRACK"\n    CUTOFF = "CUTOFF"\n    EXHAUSTED = "EXHAUSTED"\n\n\n@dataclass(frozen=True, slots=True)\nclass TraceStep:\n    iteration: int\n    event: TraceEvent\n    node_order: int\n    parent_order: int | None\n    depth: int\n    rule_id: str | None\n    state: State\n\n\n@dataclass(frozen=True, slots=True)\nclass Trace:\n    steps: tuple[TraceStep, ...] = ()\n\n    def __iter__(self) -> Iterator[TraceStep]:\n        return iter(self.steps)\n\n    def __len__(self) -> int:\n        return len(self.steps)\n\n    def of_event(self, event: TraceEvent) -> tuple[TraceStep, ...]:\n        return tuple(step for step in self.steps if step.event is event)\n\n\n@dataclass(slots=True)\nclass TraceRecorder:\n    _steps: list[TraceStep] = field(default_factory=list)\n\n    def record(\n        self,\n        *,\n        iteration: int,\n        event: TraceEvent,\n        node: Node,\n        rule_id: str | None = None,\n    ) -> None:\n        parent = node.parent\n        self._steps.append(\n            TraceStep(\n                iteration=iteration,\n                event=event,\n                node_order=node.order,\n                parent_order=None if parent is None else parent.order,\n                depth=node.depth,\n                rule_id=rule_id if rule_id is not None else _rule_id_of(node),\n                state=node.state,\n            )\n        )\n\n    def record_prune(\n        self, *, iteration: int, node: Node, rule_id: str, state: State\n    ) -> None:\n        self._steps.append(\n            TraceStep(\n                iteration=iteration,\n                event=TraceEvent.PRUNE,\n                node_order=node.order,\n                parent_order=None if node.parent is None else node.parent.order,\n                depth=node.depth,\n                rule_id=rule_id,\n                state=state,\n            )\n        )\n\n    def seal(self) -> Trace:\n        return Trace(steps=tuple(self._steps))\n\n\ndef _rule_id_of(node: Node) -> str | None:\n    return None if node.rule is None else node.rule.id\n',
    'src/core/search_tree/tree.py': 'from __future__ import annotations\n\nfrom collections.abc import Iterator\n\nfrom core.domain.state import State\nfrom core.rules.domain.base import TransitionRule\nfrom core.rules.domain.catalog import RULES\nfrom core.search_tree.node import Node\nfrom core.search_tree.path import ancestors\n\n\nclass SearchTree:\n    __slots__ = ("_nodes", "_rules")\n\n    def __init__(self, rules: tuple[TransitionRule, ...] = RULES) -> None:\n        self._rules = rules\n        self._nodes: list[Node] = []\n\n    @property\n    def rules(self) -> tuple[TransitionRule, ...]:\n        return self._rules\n\n    @property\n    def nodes(self) -> tuple[Node, ...]:\n        return tuple(self._nodes)\n\n    def root(self, state: State) -> Node:\n        self._nodes = []\n        node = Node(state=state, parent=None, rule=None, depth=0, order=0)\n        self._nodes.append(node)\n        return node\n\n    def expand(self, node: Node, rule: TransitionRule) -> Node:\n        child = Node(\n            state=rule.apply(node.state),\n            parent=node,\n            rule=rule,\n            depth=node.depth + 1,\n            order=len(self._nodes),\n            cost=node.cost + rule.cost,\n        )\n        self._nodes.append(child)\n        return child\n\n    def ancestors(self, node: Node) -> Iterator[Node]:\n        return ancestors(node)\n',
    'src/core/algorithms/domain/base.py': 'from __future__ import annotations\n\nfrom abc import ABC, abstractmethod\nfrom dataclasses import dataclass, field\nfrom time import perf_counter\nfrom typing import ClassVar\n\nfrom config import settings\nfrom core.domain.problem import Problem\nfrom core.domain.state import State\nfrom core.rules.domain.base import TransitionRule\nfrom core.rules.strategies.domain.base import ControlStrategy\nfrom core.search_tree import path\nfrom core.search_tree.metrics import MetricsCollector\nfrom core.search_tree.node import Node\nfrom core.search_tree.outcome import Outcome\nfrom core.search_tree.result import SearchResult\nfrom core.search_tree.trace import TraceEvent, TraceRecorder\nfrom core.search_tree.tree import SearchTree\n\n\nclass SearchNotStartedError(Exception):\n    def __init__(self) -> None:\n        super().__init__("search context is only available during solve()")\n\n\n@dataclass(slots=True)\nclass _RunContext:\n    problem: Problem\n    root: Node\n    metrics: MetricsCollector = field(default_factory=MetricsCollector)\n    trace: TraceRecorder = field(default_factory=TraceRecorder)\n    goal_node: Node | None = None\n\n\nclass SearchAlgorithm(ABC):\n    """Motor compartilhado pelos algoritmos de busca.\n\n    Visitar, gerar, podar, contar e registrar vivem aqui. Cada algoritmo\n    implementa só o próprio laço, em `_search`.\n    """\n\n    name: ClassVar[str] = "abstract"\n\n    def __init__(\n        self,\n        tree: SearchTree,\n        strategy: ControlStrategy,\n        *,\n        max_iterations: int = settings.MAX_ITERATIONS,\n    ) -> None:\n        self._tree = tree\n        self._strategy = strategy\n        self._max_iterations = max_iterations\n        self._context: _RunContext | None = None\n\n    @property\n    def strategy(self) -> ControlStrategy:\n        return self._strategy\n\n    @property\n    def max_iterations(self) -> int:\n        return self._max_iterations\n\n    def solve(self, problem: Problem) -> SearchResult:\n        """Porta de entrada da busca.\n\n        Monta o contexto da execução, cronometra o `_search` do algoritmo\n        concreto e sela métricas, trace e caminho no resultado.\n        """\n        context = _RunContext(problem=problem, root=self._tree.root(problem.initial))\n        self._context = context\n        context.metrics.count_generated(context.root.depth)\n        context.trace.record(iteration=0, event=TraceEvent.ROOT, node=context.root)\n\n        started = perf_counter()\n        try:\n            outcome = self._search(problem)\n        finally:\n            elapsed_ms = (perf_counter() - started) * 1000.0\n            self._context = None\n\n        goal_node = context.goal_node if outcome.is_success else None\n        solution = () if goal_node is None else path.path_to(goal_node)\n        return SearchResult(\n            algorithm=self.name,\n            strategy=self._strategy.name,\n            problem=problem.id,\n            outcome=outcome,\n            solution_path=solution,\n            applied_rules=() if goal_node is None else path.applied_rules(goal_node),\n            metrics=context.metrics.seal(elapsed_ms),\n            trace=context.trace.seal(),\n        )\n\n    @abstractmethod\n    def _search(self, problem: Problem) -> Outcome:\n        """O laço de cada algoritmo. A única coisa que eles não compartilham."""\n        ...\n\n    def _applicable_rules(self, node: Node) -> tuple[TransitionRule, ...]:\n        """As regras que este nó pode usar, prontas para serem tentadas.\n\n        Faz os três passos em ordem: testa quais são aplicáveis, ordena pela\n        estratégia e poda as que repetiriam um estado.\n        """\n        context = self._require_context()\n        applicable: list[TransitionRule] = []\n        for rule in self._tree.rules:\n            context.metrics.count_rule_test()\n            if rule.is_applicable(node.state):\n                applicable.append(rule)\n\n        allowed: list[TransitionRule] = []\n        for rule in self._strategy.order(tuple(applicable)):\n            successor = rule.apply(node.state)\n            if self._is_repetition(node, rule, successor):\n                context.trace.record_prune(\n                    iteration=context.metrics.iterations,\n                    node=node,\n                    rule_id=rule.id,\n                    state=successor,\n                )\n                continue\n            allowed.append(rule)\n        return tuple(allowed)\n\n    def _is_repetition(\n        self, node: Node, rule: TransitionRule, successor: State\n    ) -> bool:\n        """A política de repetição: aqui, um estado já presente no caminho.\n\n        A busca em largura sobrescreve para usar o conjunto global de\n        fechados, que é uma poda mais forte. A ordenada sobrescreve para\n        comparar o custo do novo caminho com o menor já conhecido.\n        """\n        return path.contains_state(node, successor)\n\n    def _require_context(self) -> _RunContext:\n        """O contexto da execução. Só existe durante o `solve`."""\n        if self._context is None:\n            raise SearchNotStartedError\n        return self._context\n\n    def _next_iteration(self) -> bool:\n        """Conta mais uma iteração e aplica o teto.\n\n        Devolve `False` quando o limite estourou, e aí a busca encerra em\n        LIMITE. Chame no topo do laço, antes de qualquer trabalho.\n        """\n        context = self._require_context()\n        if context.metrics.iterations >= self._max_iterations:\n            context.trace.record(\n                iteration=context.metrics.iterations,\n                event=TraceEvent.CUTOFF,\n                node=context.root,\n            )\n            return False\n        context.metrics.count_iteration()\n        return True\n\n    def _visit(self, node: Node) -> None:\n        """Só contabilidade: registra que o nó foi olhado. Uma vez por nó."""\n        context = self._require_context()\n        context.metrics.count_visited()\n        context.trace.record(\n            iteration=context.metrics.iterations, event=TraceEvent.VISIT, node=node\n        )\n\n    def _expand(self, node: Node, rule: TransitionRule) -> Node:\n        """Aplica a regra e cria o nó filho.\n\n        É o único ponto do motor que faz a árvore crescer.\n        """\n        context = self._require_context()\n        child = self._tree.expand(node, rule)\n        context.metrics.count_generated(child.depth)\n        context.trace.record(\n            iteration=context.metrics.iterations, event=TraceEvent.GENERATE, node=child\n        )\n        return child\n\n    def _succeed(self, node: Node) -> Outcome:\n        """Guarda o nó objetivo, de onde o caminho solução é reconstruído."""\n        context = self._require_context()\n        context.goal_node = node\n        context.trace.record(\n            iteration=context.metrics.iterations, event=TraceEvent.GOAL, node=node\n        )\n        return Outcome.SUCCESS\n\n    def _deadlock(self, node: Node) -> None:\n        """Só contabilidade: este nó nasceu sem saída.\n\n        Não encerra busca alguma. Quem decide o que fazer com o impasse é o\n        algoritmo: a irrevogável desiste, o backtracking retrocede.\n        """\n        context = self._require_context()\n        context.metrics.count_deadlock()\n        context.trace.record(\n            iteration=context.metrics.iterations, event=TraceEvent.DEADLOCK, node=node\n        )\n\n    def _backtrack(self, node: Node) -> None:\n        """Só contabilidade: um retrocesso aconteceu.\n\n        Nada é desfeito aqui. O retrocesso em si é o algoritmo deixando de\n        devolver o nó à pilha.\n        """\n        context = self._require_context()\n        context.metrics.count_backtrack()\n        context.trace.record(\n            iteration=context.metrics.iterations, event=TraceEvent.BACKTRACK, node=node\n        )\n\n    def _exhausted(self) -> Outcome:\n        """Fim de linha: nada mais a explorar e nenhum objetivo encontrado."""\n        context = self._require_context()\n        context.trace.record(\n            iteration=context.metrics.iterations,\n            event=TraceEvent.EXHAUSTED,\n            node=context.root,\n        )\n        return Outcome.FAILURE\n',
    'src/core/algorithms/domain/registry.py': 'from __future__ import annotations\n\nfrom collections.abc import Mapping\nfrom types import MappingProxyType\n\nfrom core.algorithms.backtracking import BacktrackingSearch\nfrom core.algorithms.breadth_first import BreadthFirstSearch\nfrom core.algorithms.domain.base import SearchAlgorithm\nfrom core.algorithms.irrevocable import IrrevocableSearch\nfrom core.algorithms.ordered import OrderedSearch\n\n\nclass UnknownAlgorithmError(Exception):\n    def __init__(self, name: str) -> None:\n        super().__init__(f"unknown search algorithm: {name!r}")\n        self.name = name\n\n\nALGORITHMS: Mapping[str, type[SearchAlgorithm]] = MappingProxyType(\n    {\n        IrrevocableSearch.name: IrrevocableSearch,\n        BacktrackingSearch.name: BacktrackingSearch,\n        BreadthFirstSearch.name: BreadthFirstSearch,\n        OrderedSearch.name: OrderedSearch,\n    }\n)\n\nALGORITHM_NAMES: tuple[str, ...] = tuple(ALGORITHMS)\n\n\ndef get_algorithm(name: str) -> type[SearchAlgorithm]:\n    try:\n        return ALGORITHMS[name]\n    except KeyError:\n        raise UnknownAlgorithmError(name) from None\n',
    'src/core/algorithms/backtracking.py': '"""BUSCA COM BACKTRACKING\n\n1. Estado inicial e objetivo (carta P1)\n\n    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])\n\n2. Critério\n\n    A mesma descida da busca irrevogável, mas as regras válidas de cada\n    estado ficam guardadas numa pilha. Quando um estado não tem mais regra,\n    ele sai da pilha (retrocesso) e o estado anterior tenta a próxima regra\n    que tinha guardado. Uma regra que leva a um estado já no caminho atual\n    é descartada (poda).\n\n3. Execução com a ordem decrescente R6 → R5 → R4 → R3 → R2 → R1\n\n    Passo 1   S0 = ([V, R], [A], [])\n              guarda [R4, R3, R2, R1], aplica R4\n              ↓\n    Passo 2   S1 = ([V, R], [], [A])\n              R6 volta a S0: poda.  Guarda [R5, R1], aplica R5\n              ↓\n    Passo 3   S2 = ([V, R, A], [], [])\n              R2 volta a S1 e R1 volta a S0: poda.  Nada guardado.\n              ✗ impasse → retrocesso: S2 sai da pilha\n              ↑\n    Passo 4   S1 = ([V, R], [], [A])   ainda tem [R1], aplica R1\n              ↓\n    Passo 5   S3 = ([V], [R], [A])\n              R3 volta a S1: poda.  Guarda [R6, R5, R1], aplica R6\n              ↓\n              ...  continua descendo; a cada beco sem saída, um retrocesso\n\n    Passo 51  S36 = ([], [R, V], [A])\n              ✓ SUCESSO\n\n4. Árvore de busca (início)\n\n    S0 ([V,R], [A], [])\n    └── R4 → S1 ([V,R], [], [A])\n        ├── R5 → S2 ([V,R,A], [], [])   (impasse, retrocesso)\n        └── R1 → S3 ([V], [R], [A])\n            └── R6 → S4 ([V], [R,A], [])\n                └── R3 → S5 ([V,A], [R], [])\n                    └── R4 → S6 ([V,A], [], [R])\n                        ├── R5 → S7 ([V,A,R], [], [])   (impasse, retrocesso)\n                        └── R1 → S8 ([V], [A], [R])\n                            └── ...  até S36 = Sf\n\n5. Caminho encontrado\n\n    R4 R1 R6 R3 R4 R1 R6 R2 R3 R6 R3 R4 R1 R6 R2 R3 R6 R3 R4 R1 R6 R2\n\n    Número de movimentos: 22       Iterações: 51\n    Retrocessos: 14                Impasses: 8\n\n    Com a ordem crescente: 14 movimentos em 15 iterações, sem retrocesso.\n    O caminho ótimo tem 3.\n\n6. Conclusão\n\n    O impasse da busca irrevogável vira um desvio: nenhum ramo é abandonado\n    sem ser explorado e, como todo estado alcança todo estado, a solução\n    sempre aparece. Só não é a melhor. A busca para no primeiro objetivo\n    que encontra, tenha o caminho o comprimento que tiver.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import deque\nfrom typing import ClassVar\n\nfrom core.algorithms.domain.base import SearchAlgorithm\nfrom core.domain.problem import Problem\nfrom core.rules.domain.base import TransitionRule\nfrom core.search_tree.frontier import StackFrontier\nfrom core.search_tree.node import Node\nfrom core.search_tree.outcome import Outcome\n\n\nclass BacktrackingSearch(SearchAlgorithm):\n    name: ClassVar[str] = "backtracking"\n\n    def _search(self, problem: Problem) -> Outcome:\n        """Desce enquanto houver regra; sem regra, retrocede.\n\n        Cada iteração olha a ponta do caminho e faz uma coisa OU a outra:\n        desce, e o nó continua no caminho com um filho por cima; ou retrocede,\n        e o nó sai do caminho, deixando o pai como ponta.\n\n        Sempre encontra solução, se existir — mas é a primeira que aparecer,\n        não a mais curta.\n        """\n        context = self._require_context()\n\n        path = StackFrontier()\n        path.push(context.root)\n\n        # Para cada nó do caminho, as regras que ele ainda não tentou. Sem\n        # isso não há retrocesso: o nó não saberia por onde continuar.\n        untried: dict[int, deque[TransitionRule]] = {}\n\n        while len(path):\n            if not self._next_iteration():\n                return Outcome.CUTOFF\n\n            # Topo da pilha = ponta do caminho. Ele só volta para a pilha se\n            # ainda tiver regra -- ver o if lá embaixo.\n            node = path.pop()\n\n            if problem.is_goal(node.state):\n                return self._succeed(node)\n\n            remaining = self._remaining_rules(node, untried)\n            if remaining:\n                # DESCE: o nó continua no caminho e o filho entra por cima.\n                path.push(node)\n                path.push(self._expand(node, remaining.popleft()))\n            else:\n                # RETROCEDE: sem regras, o nó não volta e sai do caminho. Na\n                # próxima iteração o topo é o pai, que tenta a regra seguinte.\n                self._backtrack(node)\n\n        return self._exhausted()\n\n    def _remaining_rules(\n        self, node: Node, untried: dict[int, deque[TransitionRule]]\n    ) -> deque[TransitionRule]:\n        """Na primeira visita monta a lista do nó; nas voltas seguintes apenas\n        devolve o que sobrou dela. _applicable_rules já poda a regra que\n        repetiria um estado do caminho.\n\n        As seis regras que entram nessa lista:\n\n        ```\n        regra | de | para | inversa\n        ------+----+------+--------\n        R1    | H1 | H2   | R3\n        R2    | H1 | H3   | R5\n        R3    | H2 | H1   | R1\n        R4    | H2 | H3   | R6\n        R5    | H3 | H1   | R2\n        R6    | H3 | H2   | R4\n        ```\n\n        Toda regra tem inversa exata, e é daí que vem a necessidade da poda:\n        sem ela, R1 seguido de R3 devolveria a busca ao mesmo estado, sempre.\n        """\n        if node.order not in untried:\n            self._visit(node)\n            candidates = self._applicable_rules(node)\n            untried[node.order] = deque(candidates)\n            if not candidates:\n                self._deadlock(node)\n        return untried[node.order]\n',
    'src/core/algorithms/irrevocable.py': '"""BUSCA IRREVOGÁVEL\n\n1. Estado inicial e objetivo (carta P1)\n\n    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])\n\n       H1      H2      H3              H1      H2      H3\n      ┌───┐   ┌───┐   ┌───┐           ┌───┐   ┌───┐   ┌───┐\n      │ R │   │   │   │   │           │   │   │ V │   │   │\n      │ V │   │ A │   │   │           │   │   │ R │   │ A │\n      └───┘   └───┘   └───┘           └───┘   └───┘   └───┘\n\n2. Regras válidas em S0\n\n    R1 = H1 → H2  ✓          R4 = H2 → H3  ✓\n    R2 = H1 → H3  ✓          R5 = H3 → H1  ✗  H3 vazia\n    R3 = H2 → H1  ✓          R6 = H3 → H2  ✗  H3 vazia\n\n3. Critério\n\n    A estratégia ordena as regras válidas e a busca aplica a primeira.\n    Não guarda as outras: cada passo é definitivo e não há retorno.\n    Uma regra que leva a um estado já percorrido é descartada (poda).\n\n4. Execução com a ordem decrescente R6 → R5 → R4 → R3 → R2 → R1\n\n    Passo 1   S0 = ([V, R], [A], [])\n              válidas em ordem: R4, R3, R2, R1\n              aplica R4 (azul: H2 → H3)\n              ↓\n    Passo 2   S1 = ([V, R], [], [A])\n              válidas em ordem: R6, R5, R1\n              R6 volta a S0: poda.  Aplica R5 (azul: H3 → H1)\n              ↓\n    Passo 3   S2 = ([V, R, A], [], [])\n              válidas em ordem: R2, R1\n              R2 volta a S1: poda.  R1 volta a S0: poda.\n              Nenhuma regra sobrou.\n              ↓\n              ✗ IMPASSE\n\n5. Árvore de busca\n\n    S0 ([V,R], [A], [])\n    └── R4 → S1 ([V,R], [], [A])\n        └── R5 → S2 ([V,R,A], [], [])   (impasse)\n\n6. A mesma carta com a ordem crescente R1 → R2 → R3 → R4 → R5 → R6\n\n    S0 ([V,R],[A],[]) → R1 → R2 → R3 → R3 → R5 → R1 → R1 → R2 → R3 → R3\n       → R5 → R1 → R1 → R2 → Sf ([],[R,V],[A])\n\n    ✓ SUCESSO com 14 movimentos.  O caminho ótimo tem 3.\n\n7. Conclusão\n\n    O caminho nunca repete estado, então a busca sempre para, no máximo\n    depois dos 36 estados. Parar não é chegar: a ordem decrescente termina\n    em impasse e a crescente só chega depois de 14 movimentos.\n    É o único método em que a estratégia decide o desfecho.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import ClassVar\n\nfrom core.algorithms.domain.base import SearchAlgorithm\nfrom core.domain.problem import Problem\nfrom core.search_tree.outcome import Outcome\n\n\nclass IrrevocableSearch(SearchAlgorithm):\n    name: ClassVar[str] = "irrevocable"\n\n    def _search(self, problem: Problem) -> Outcome:\n        node = self._require_context().root\n        while True:\n            if not self._next_iteration():\n                return Outcome.CUTOFF\n            if problem.is_goal(node.state):\n                return self._succeed(node)\n\n            self._visit(node)\n            candidates = self._applicable_rules(node)\n            if not candidates:\n                self._deadlock(node)\n                return Outcome.DEADLOCK\n\n            node = self._expand(node, candidates[0])\n',
    'src/core/algorithms/ordered.py': '"""BUSCA ORDENADA\n\n1. Estado inicial e objetivo (carta P1)\n\n    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])\n\n2. Custos\n\n    Cada regra custa a distância entre as hastes que ela liga:\n\n        R1 = H1 → H2: 1     R3 = H2 → H1: 1     R4 = H2 → H3: 1\n        R2 = H1 → H3: 2     R5 = H3 → H1: 2     R6 = H3 → H2: 1\n\n    O custo de um nó é a soma das regras do caminho da raiz até ele.\n\n3. Critério\n\n    ABERTOS é ordenada pelo MENOR custo. No empate, sai o nó gerado primeiro;\n    entre irmãos de mesmo custo, decide a estratégia (aqui, a ordem crescente\n    R1 → R2 → R3 → R4 → R5 → R6). A busca só encerra quando o objetivo vira\n    o estado atual, não quando ele é gerado.\n\n    Poda pelo vetor de menor custo: o filho cujo estado já foi gerado com\n    custo menor ou igual é descartado. Se o novo custo for menor, o nó\n    antigo sai de ABERTOS e da árvore e o novo entra no lugar.\n\n4. Execução\n\n    Expansão de S0 = ([V, R], [A], []), custo 0\n        R1 → S1 = ([V], [A, R], [])      1\n        R2 → S2 = ([V], [A], [R])        2\n        R3 → S3 = ([V, R, A], [], [])    1\n        R4 → S4 = ([V, R], [], [A])      1\n        R5, R6: inválidas, H3 vazia\n        ABERTOS  = [S1(1), S3(1), S4(1), S2(2)]\n        FECHADOS = [S0(0)]\n\n    Expansão de S1 = ([V], [A, R], []), custo 1\n        R2 → S5 = ([], [A, R], [V])      3\n        R3 → S0 (2 ≥ 0): poda\n        R4 → estado de S2 (2 ≥ 2): poda\n        ABERTOS  = [S3(1), S4(1), S2(2), S5(3)]\n        FECHADOS = [S0(0), S1(1)]\n\n    Expansão de S3 = ([V, R, A], [], []), custo 1\n        R1 → S0, R2 → S4: poda.  Nada novo.\n        ✗ impasse: S3 só vai para FECHADOS\n        ABERTOS  = [S4(1), S2(2), S5(3)]\n        FECHADOS = [S0(0), S1(1), S3(1)]\n\n    Expansão de S4 = ([V, R], [], [A]), custo 1\n        R1 → S6 = ([V], [R], [A])        2\n        R5 → S3, R6 → S0: poda\n        ABERTOS  = [S2(2), S6(2), S5(3)]\n        FECHADOS = [S0(0), S1(1), S3(1), S4(1)]\n\n    Expansão de S2 = ([V], [A], [R]), custo 2\n        R1 → S7 = ([], [A, V], [R])      3\n        R3 → S8 = ([V, A], [], [R])      3\n        R5 → S0, R6 → S1: poda\n        ABERTOS  = [S6(2), S5(3), S7(3), S8(3)]\n        FECHADOS = [S0(0), S1(1), S3(1), S4(1), S2(2)]\n\n    Expansão de S6 = ([V], [R], [A]), custo 2\n        R1 → S9  = ([], [R, V], [A])     3   ← é Sf, mas só entra na fila\n        R5 → S10 = ([V, A], [R], [])     4\n        R6 → S11 = ([V], [R, A], [])     3\n        R3 → S4: poda\n        ABERTOS  = [S5(3), S7(3), S8(3), S9(3), S11(3), S10(4)]\n        FECHADOS = [S0(0), S1(1), S3(1), S4(1), S2(2), S6(2)]\n\n    S5, S7 e S8 custam 3 como S9, mas foram gerados antes e saem primeiro.\n        S5 gera S12(4).  S7 gera S13(5).  S8 gera S14(5).\n        ABERTOS  = [S9(3), S11(3), S10(4), S12(4), S13(5), S14(5)]\n        FECHADOS = [S0(0), S1(1), S3(1), S4(1), S2(2), S6(2), S5(3),\n                    S7(3), S8(3)]\n\n    Chega a vez de S9 = ([], [R, V], [A])\n        ✓ SUCESSO\n\n5. Árvore de busca (custo acumulado entre parênteses)\n\n    S0 ([V,R], [A], [])  (0)\n    ├── R1 → S1 ([V], [A,R], [])  (1)\n    │   └── R2 → S5 ([], [A,R], [V])  (3)\n    │       └── R3 → S12  (4)\n    ├── R2 → S2 ([V], [A], [R])  (2)\n    │   ├── R1 → S7 ([], [A,V], [R])  (3)\n    │   │   └── R5 → S13  (5)\n    │   └── R3 → S8 ([V,A], [], [R])  (3)\n    │       └── R5 → S14  (5)\n    ├── R3 → S3 ([V,R,A], [], [])  (1)   (impasse)\n    └── R4 → S4 ([V,R], [], [A])  (1)\n        └── R1 → S6 ([V], [R], [A])  (2)\n            ├── R1 → S9 ([], [R,V], [A])  (3)   (objetivo)\n            ├── R5 → S10 ([V,A], [R], [])  (4)\n            └── R6 → S11 ([V], [R,A], [])  (3)\n\n6. Caminho solução\n\n    S0 ([V,R], [A], [])\n     │ R4: azul H2 → H3        custo 1\n     ↓\n    S4 ([V,R], [], [A])\n     │ R1: vermelho H1 → H2    custo 1\n     ↓\n    S6 ([V], [R], [A])\n     │ R1: verde H1 → H2       custo 1\n     ↓\n    Sf ([], [R,V], [A])\n\n    Custo da solução: 1 + 1 + 1 = 3\n    Número de movimentos: 3      Iterações: 10      Gerados: 15\n\n    Com as ordens decrescente e custom: o mesmo caminho em 11 iterações.\n\n7. Conclusão\n\n    Um nó só é expandido depois de todos os mais baratos, então quando o\n    objetivo vira o estado atual nenhum nó aberto chega nele mais barato:\n    a solução é a de MENOR CUSTO, que nem sempre é a de menos movimentos.\n    Com custo unitário ela seria a busca em largura. Aqui, em P1, as duas\n    acham o mesmo caminho, mas a ordenada deixa para depois os ramos que\n    começam por R2 e chega em 10 iterações contra 14.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import ClassVar\n\nfrom config import settings\nfrom core.algorithms.domain.base import SearchAlgorithm\nfrom core.domain.problem import Problem\nfrom core.domain.state import State\nfrom core.rules.domain.base import TransitionRule\nfrom core.rules.strategies.domain.base import ControlStrategy\nfrom core.search_tree.frontier import PriorityFrontier\nfrom core.search_tree.node import Node\nfrom core.search_tree.outcome import Outcome\nfrom core.search_tree.tree import SearchTree\n\n\nclass OrderedSearch(SearchAlgorithm):\n    name: ClassVar[str] = "ordered"\n\n    def __init__(\n        self,\n        tree: SearchTree,\n        strategy: ControlStrategy,\n        *,\n        max_iterations: int = settings.MAX_ITERATIONS,\n    ) -> None:\n        super().__init__(tree, strategy, max_iterations=max_iterations)\n        self._best_cost: dict[State, int] = {}\n        self._open: dict[State, Node] = {}\n\n    def _search(self, problem: Problem) -> Outcome:\n        """Sempre expande o nó aberto de menor custo acumulado.\n\n        O objetivo só encerra a busca quando vira o estado atual, não quando\n        é gerado: só aí nenhum nó aberto pode chegar nele mais barato.\n        """\n        context = self._require_context()\n        frontier = PriorityFrontier()\n        frontier.push(context.root)\n\n        # Vetor de menor custo: o custo do caminho mais barato já gerado até\n        # cada estado. Cobre ABERTOS e FECHADOS de uma vez.\n        self._best_cost = {context.root.state: 0}\n        # O nó de cada estado que ainda espera em ABERTOS, para poder tirá-lo\n        # de lá quando um caminho mais barato até o mesmo estado aparecer.\n        self._open = {context.root.state: context.root}\n\n        while len(frontier):\n            if not self._next_iteration():\n                return Outcome.CUTOFF\n\n            node = frontier.pop()\n            del self._open[node.state]\n            if problem.is_goal(node.state):\n                return self._succeed(node)\n\n            self._visit(node)\n            candidates = self._applicable_rules(node)\n            if not candidates:\n                self._deadlock(node)\n\n            for rule in candidates:\n                child = self._expand(node, rule)\n                older = self._open.get(child.state)\n                if older is not None:\n                    # Chegou mais barato a um estado que ainda está aberto:\n                    # o nó antigo sai de ABERTOS e da árvore.\n                    frontier.remove(older)\n                    self._discard(older)\n                self._best_cost[child.state] = child.cost\n                self._open[child.state] = child\n                frontier.push(child)\n\n        return self._exhausted()\n\n    def _is_repetition(\n        self, node: Node, rule: TransitionRule, successor: State\n    ) -> bool:\n        """Poda o filho que não melhora o menor custo conhecido do estado.\n\n        Com custos positivos, um estado já fechado nunca é alcançado mais\n        barato depois, então ele sempre cai aqui.\n        """\n        best = self._best_cost.get(successor)\n        return best is not None and node.cost + rule.cost >= best\n\n    def _discard(self, node: Node) -> None:\n        """Só contabilidade: registra o nó substituído como poda do pai."""\n        context = self._require_context()\n        parent, rule = node.parent, node.rule\n        assert parent is not None and rule is not None\n        context.trace.record_prune(\n            iteration=context.metrics.iterations,\n            node=parent,\n            rule_id=rule.id,\n            state=node.state,\n        )\n',
    'src/libs/inputs/exceptions.py': 'from __future__ import annotations\n\n\nclass InputError(Exception): ...\n\n\nclass InvalidArgumentError(InputError):\n    def __init__(self, detail: str) -> None:\n        super().__init__(f"invalid argument: {detail}")\n        self.detail = detail\n\n\nclass UnknownSelectionError(InputError):\n    def __init__(self, kind: str, name: str, available: tuple[str, ...]) -> None:\n        super().__init__(f"unknown {kind}: {name!r}")\n        self.kind = kind\n        self.name = name\n        self.available = available\n',
    'src/libs/inputs/parser.py': 'from __future__ import annotations\n\nimport argparse\nfrom collections.abc import Iterable, Sequence\nfrom pathlib import Path\nfrom typing import NoReturn\n\nfrom config import settings\nfrom core.algorithms.domain.registry import ALGORITHM_NAMES\nfrom core.domain.problem import PROBLEM_IDS\nfrom core.rules.strategies.custom_order import parse_order\nfrom core.rules.strategies.domain.registry import STRATEGY_NAMES\nfrom libs.inputs.exceptions import InvalidArgumentError\nfrom libs.inputs.selection import ExecutionRequest\n\n\nclass _HelpFormatter(argparse.HelpFormatter):\n    def add_usage(\n        self,\n        usage: str | None,\n        actions: Iterable[argparse.Action],\n        groups: Iterable[argparse._MutuallyExclusiveGroup],\n        prefix: str | None = None,\n    ) -> None:\n        super().add_usage(usage, actions, groups, settings.USAGE_PREFIX)\n\n\nclass _Parser(argparse.ArgumentParser):\n    def error(self, message: str) -> NoReturn:\n        raise InvalidArgumentError(message)\n\n    def exit(self, status: int = 0, message: str | None = None) -> NoReturn:\n        raise InvalidArgumentError(message or "")\n\n\ndef build_parser() -> argparse.ArgumentParser:\n    parser = _Parser(\n        prog=settings.PROGRAM_NAME,\n        description=settings.DESCRIPTION,\n        epilog=settings.EPILOG,\n        add_help=False,\n        formatter_class=_HelpFormatter,\n    )\n    group = parser.add_argument_group(settings.OPTIONS_TITLE)\n    group.add_argument("-h", "--help", action="store_true", help=settings.HELP_HELP)\n    group.add_argument("--problem", action="append", help=settings.HELP_PROBLEM)\n    group.add_argument("--algorithm", action="append", help=settings.HELP_ALGORITHM)\n    group.add_argument("--strategy", action="append", help=settings.HELP_STRATEGY)\n    group.add_argument("--all", action="store_true", help=settings.HELP_ALL)\n    group.add_argument("--all-goals", action="store_true", help=settings.HELP_ALL_GOALS)\n    group.add_argument("--order", help=settings.HELP_ORDER)\n    group.add_argument("--show-tree", action="store_true", help=settings.HELP_SHOW_TREE)\n    group.add_argument(\n        "--show-trace", action="store_true", help=settings.HELP_SHOW_TRACE\n    )\n    group.add_argument(\n        "--show-states", action="store_true", help=settings.HELP_SHOW_STATES\n    )\n    group.add_argument(\n        "--format", default=settings.DEFAULT_OUTPUT_FORMAT, help=settings.HELP_FORMAT\n    )\n    group.add_argument("--output", help=settings.HELP_OUTPUT)\n    group.add_argument("--svg-dir", help=settings.HELP_SVG_DIR)\n    group.add_argument(\n        "--rank-mode", default=settings.DEFAULT_RANK_MODE, help=settings.HELP_RANK_MODE\n    )\n    group.add_argument(\n        "--max-iterations",\n        default=str(settings.MAX_ITERATIONS),\n        help=settings.HELP_MAX_ITERATIONS,\n    )\n    group.add_argument("--seed", help=settings.HELP_SEED)\n    group.add_argument("--list", action="store_true", help=settings.HELP_LIST)\n    group.add_argument("--verbose", action="store_true", help=settings.HELP_VERBOSE)\n    return parser\n\n\ndef help_text() -> str:\n    return build_parser().format_help()\n\n\ndef parse(argv: Sequence[str]) -> ExecutionRequest:\n    namespace = build_parser().parse_args(list(argv))\n\n    select_all: bool = namespace.all\n    problems = _selected(namespace.problem, PROBLEM_IDS, select_all, upper=True)\n    algorithms = _selected(namespace.algorithm, ALGORITHM_NAMES, select_all)\n    strategies = _selected(namespace.strategy, STRATEGY_NAMES, select_all)\n    order = parse_order(namespace.order) if namespace.order else None\n\n    return ExecutionRequest(\n        problem_ids=problems,\n        algorithm_names=algorithms,\n        strategy_names=strategies,\n        custom_order=order,\n        show_tree=namespace.show_tree,\n        show_trace=namespace.show_trace,\n        show_states=namespace.show_states,\n        output_format=namespace.format.lower(),\n        rank_mode=namespace.rank_mode.lower(),\n        max_iterations=_positive_int("--max-iterations", namespace.max_iterations),\n        output_path=Path(namespace.output) if namespace.output else None,\n        svg_dir=Path(namespace.svg_dir) if namespace.svg_dir else None,\n        list_only=namespace.list,\n        show_help=namespace.help,\n        verbose=namespace.verbose,\n        all_goals=namespace.all_goals,\n        seed=None\n        if namespace.seed is None\n        else _positive_int("--seed", namespace.seed),\n    )\n\n\ndef _selected(\n    raw: list[str] | None,\n    available: tuple[str, ...],\n    select_all: bool,\n    *,\n    upper: bool = False,\n) -> tuple[str, ...]:\n    if select_all or not raw:\n        return available\n    names = [\n        token.strip().upper() if upper else token.strip().lower()\n        for entry in raw\n        for token in entry.split(",")\n        if token.strip()\n    ]\n    return tuple(dict.fromkeys(names))\n\n\ndef _positive_int(flag: str, raw: str) -> int:\n    try:\n        value = int(raw)\n    except ValueError:\n        raise InvalidArgumentError(f"{flag}={raw!r}") from None\n    if value <= 0:\n        raise InvalidArgumentError(f"{flag}={raw!r}")\n    return value\n',
    'src/libs/inputs/selection.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nfrom config import settings\n\n\n@dataclass(frozen=True, slots=True)\nclass ExecutionRequest:\n    problem_ids: tuple[str, ...]\n    algorithm_names: tuple[str, ...]\n    strategy_names: tuple[str, ...]\n    custom_order: tuple[str, ...] | None = None\n    show_tree: bool = False\n    show_trace: bool = False\n    show_states: bool = False\n    output_format: str = settings.DEFAULT_OUTPUT_FORMAT\n    rank_mode: str = settings.DEFAULT_RANK_MODE\n    max_iterations: int = settings.MAX_ITERATIONS\n    output_path: Path | None = None\n    svg_dir: Path | None = None\n    list_only: bool = False\n    show_help: bool = False\n    verbose: bool = False\n    seed: int | None = None\n    all_goals: bool = False\n\n    @property\n    def execution_count(self) -> int:\n        return (\n            len(self.problem_ids) * len(self.algorithm_names) * len(self.strategy_names)\n        )\n',
    'src/libs/inputs/validator.py': 'from __future__ import annotations\n\nfrom collections.abc import Iterable\n\nfrom config import settings\nfrom core.algorithms.domain.registry import ALGORITHM_NAMES\nfrom core.domain.problem import PROBLEM_IDS\nfrom core.rules.strategies.domain.registry import STRATEGY_NAMES\nfrom libs.inputs.exceptions import UnknownSelectionError\nfrom libs.inputs.selection import ExecutionRequest\n\n\ndef validate(request: ExecutionRequest) -> ExecutionRequest:\n    _require_known("problem", request.problem_ids, PROBLEM_IDS)\n    _require_known("algorithm", request.algorithm_names, ALGORITHM_NAMES)\n    _require_known("strategy", request.strategy_names, STRATEGY_NAMES)\n    _require_known("format", (request.output_format,), settings.OUTPUT_FORMATS)\n    _require_known("rank_mode", (request.rank_mode,), settings.RANK_MODES)\n    return request\n\n\ndef _require_known(\n    kind: str, selected: Iterable[str], available: tuple[str, ...]\n) -> None:\n    known = set(available)\n    for name in selected:\n        if name not in known:\n            raise UnknownSelectionError(kind, name, available)\n',
    'src/libs/outputs/console.py': 'from __future__ import annotations\n\nimport sys\nimport textwrap\nfrom typing import TextIO\n\nfrom core.search_tree.result import SearchResult\nfrom libs.outputs import state_render, theme, trace_render, tree_render\nfrom libs.outputs.formatter import ProblemReport, Report\nfrom libs.outputs.trace_render import render_table\nfrom libs.ranking.leaderboard import Leaderboard, Stat, Summary, SummaryRow\n\n\nclass ConsoleFormatter:\n    name = "console"\n\n    def render(self, report: Report) -> str:\n        blocks = (\n            []\n            if report.summary_only\n            else [self._render_problem(entry, report) for entry in report.problems]\n        )\n        if report.summary is not None and (\n            report.summary_only or len(report.problems) > 1\n        ):\n            blocks.append(self._render_summary(report.summary))\n        return "\\n\\n".join(block for block in blocks if block)\n\n    def _render_summary(self, summary: Summary) -> str:\n        sections = [render_section(theme.SUMMARY_HEADER, render_summary(summary))]\n        if summary.highlights:\n            sections.append(\n                render_section(theme.HIGHLIGHTS_HEADER, render_highlights(summary))\n            )\n        return "\\n\\n".join(sections)\n\n    def _render_problem(self, entry: ProblemReport, report: Report) -> str:\n        sections = [\n            f"{theme.PROBLEM_HEADER} {entry.problem_id}",\n            "",\n            f"{theme.INITIAL_STATE_HEADER}: {state_render.render_inline(entry.initial_state)}",\n            f"{theme.GOAL_STATE_HEADER}: {state_render.render_inline(entry.goal_state)}",\n        ]\n        for result in entry.results:\n            sections.append("")\n            sections.append(render_result_box(result))\n            sections.extend(_render_details(result, report))\n        if entry.leaderboard is not None:\n            sections.extend(\n                [\n                    "",\n                    render_section(\n                        theme.LEADERBOARD_HEADER, render_leaderboard(entry.leaderboard)\n                    ),\n                ]\n            )\n        return "\\n".join(sections)\n\n\ndef render_result_box(result: SearchResult) -> str:\n    metrics = result.metrics\n    moves = (\n        theme.ABSENT if result.solution_length is None else str(result.solution_length)\n    )\n    cost = theme.ABSENT if result.solution_cost is None else str(result.solution_cost)\n    path = (\n        theme.ARROW.join(result.applied_rules) if result.applied_rules else theme.ABSENT\n    )\n    rows = (\n        (theme.REPORT_LABELS["outcome"], theme.outcome_label(result.outcome)),\n        (theme.REPORT_LABELS["moves"], moves),\n        (theme.REPORT_LABELS["cost"], cost),\n        (theme.REPORT_LABELS["path"], path),\n        (theme.REPORT_LABELS["iterations"], str(metrics.iterations)),\n        (theme.REPORT_LABELS["nodes_generated"], str(metrics.nodes_generated)),\n        (theme.REPORT_LABELS["nodes_visited"], str(metrics.nodes_visited)),\n        (theme.REPORT_LABELS["rules_tested"], str(metrics.rules_tested)),\n        (theme.REPORT_LABELS["backtracks"], str(metrics.backtracks)),\n        (theme.REPORT_LABELS["deadlocks"], str(metrics.deadlocks)),\n        (theme.REPORT_LABELS["max_depth"], str(metrics.max_depth)),\n        (theme.REPORT_LABELS["elapsed"], f"{metrics.elapsed_ms:.3f}"),\n    )\n    lines = [line for label, value in rows for line in _wrap_row(label, value)]\n    return render_box(result.label, tuple(lines))\n\n\ndef _wrap_row(label: str, value: str) -> list[str]:\n    available = theme.MAX_BOX_WIDTH - 2 - theme.LABEL_WIDTH\n    chunks = textwrap.wrap(value, width=available) or [""]\n    head = f"{label.ljust(theme.LABEL_WIDTH)}{chunks[0]}"\n    tail = [f"{\'\'.ljust(theme.LABEL_WIDTH)}{chunk}" for chunk in chunks[1:]]\n    return [head, *tail]\n\n\ndef render_leaderboard(leaderboard: Leaderboard) -> str:\n    scored = leaderboard.mode == "score"\n    headers = theme.LEADERBOARD_HEADERS if scored else theme.LEADERBOARD_HEADERS[:-1]\n    rows = []\n    for row in leaderboard.rows:\n        result = row.result\n        cells = [\n            str(row.position),\n            result.algorithm,\n            result.strategy,\n            theme.outcome_label(result.outcome),\n            theme.ABSENT\n            if result.solution_length is None\n            else str(result.solution_length),\n            str(result.metrics.iterations),\n            str(result.metrics.nodes_generated),\n        ]\n        if scored:\n            cells.append(theme.ABSENT if row.score is None else f"{row.score:.1f}")\n        rows.append(tuple(cells))\n    return render_table(headers, tuple(rows))\n\n\ndef render_summary(summary: Summary) -> str:\n    rows = tuple(\n        (\n            row.algorithm,\n            row.strategy,\n            f"{row.successes}/{row.runs}",\n            str(row.deadlocks),\n            _render_stat(row.moves),\n            _render_stat(row.cost),\n            _render_stat(row.iterations),\n            _render_stat(row.nodes_generated),\n        )\n        for row in summary.rows\n    )\n    table = render_table(theme.SUMMARY_HEADERS, rows)\n    return f"{theme.SUMMARY_LEGEND}\\n\\n{table}"\n\n\ndef render_highlights(summary: Summary) -> str:\n    strategies: dict[str, int] = {}\n    for row in summary.rows:\n        strategies[row.algorithm] = strategies.get(row.algorithm, 0) + 1\n    rows = tuple(\n        (\n            theme.CRITERION_LABELS[highlight.criterion],\n            _render_winners(highlight.by_mean, strategies),\n            _render_winners(highlight.by_median, strategies),\n        )\n        for highlight in summary.highlights\n    )\n    table = render_table(theme.HIGHLIGHTS_HEADERS, rows)\n    return f"{theme.HIGHLIGHTS_LEGEND}\\n\\n{table}"\n\n\ndef _render_winners(winners: tuple[SummaryRow, ...], strategies: dict[str, int]) -> str:\n    grouped: dict[str, list[str]] = {}\n    for row in winners:\n        grouped.setdefault(row.algorithm, []).append(row.strategy)\n    return "; ".join(\n        f"{algorithm} {theme.ALL_STRATEGIES}"\n        if len(names) == strategies[algorithm]\n        else f"{algorithm} ({\', \'.join(names)})"\n        for algorithm, names in grouped.items()\n    )\n\n\ndef _render_stat(stat: Stat | None) -> str:\n    if stat is None:\n        return theme.ABSENT\n    return f"{stat.mean:.2f} / {stat.median:g}"\n\n\ndef render_box(title: str, lines: tuple[str, ...]) -> str:\n    content_width = max((len(line) for line in lines), default=0)\n    width = min(\n        max(content_width + 2, len(title) + 6, theme.MIN_BOX_WIDTH),\n        max(theme.MAX_BOX_WIDTH, len(title) + 6, content_width + 2),\n    )\n\n    header = f"{theme.BOX_HORIZONTAL} {title} "\n    top = (\n        theme.BOX_TOP_LEFT\n        + header\n        + theme.BOX_HORIZONTAL * (width - len(header))\n        + theme.BOX_TOP_RIGHT\n    )\n    body = [\n        f"{theme.BOX_VERTICAL} {line.ljust(width - 2)} {theme.BOX_VERTICAL}"\n        for line in lines\n    ]\n    bottom = (\n        theme.BOX_BOTTOM_LEFT + theme.BOX_HORIZONTAL * width + theme.BOX_BOTTOM_RIGHT\n    )\n    return "\\n".join([top, *body, bottom])\n\n\ndef render_section(title: str, body: str) -> str:\n    return f"{title}\\n{theme.BOX_HORIZONTAL * len(title)}\\n{body}"\n\n\ndef _render_details(result: SearchResult, report: Report) -> list[str]:\n    sections: list[str] = []\n    if report.show_states and result.solution_path:\n        drawings = "\\n\\n".join(\n            state_render.render_pegs(node.state) for node in result.solution_path\n        )\n        sections.extend(["", render_section(theme.STATES_HEADER, drawings)])\n    if report.show_tree:\n        sections.extend(\n            [\n                "",\n                render_section(\n                    theme.TREE_HEADER, tree_render.render_tree(result.trace)\n                ),\n            ]\n        )\n    if report.show_trace:\n        sections.extend(\n            [\n                "",\n                render_section(\n                    theme.TRACE_HEADER, trace_render.render_trace(result.trace)\n                ),\n            ]\n        )\n    return sections\n\n\ndef write(text: str, stream: TextIO | None = None) -> None:\n    target = sys.stdout if stream is None else stream\n    target.write(f"{text}\\n")\n\n\ndef write_error(text: str) -> None:\n    sys.stderr.write(f"{text}\\n")\n',
    'src/libs/outputs/dot_render.py': 'from __future__ import annotations\n\nfrom core.search_tree.result import SearchResult\nfrom core.search_tree.trace import TraceEvent, TraceStep\nfrom libs.outputs import state_render, theme\nfrom libs.outputs.tree_render import removed_edges\n\n\ndef render_dot(result: SearchResult) -> str:\n    trace = result.trace\n    root_steps = trace.of_event(TraceEvent.ROOT)\n    if not root_steps:\n        return ""\n\n    goal_orders = {step.node_order for step in trace.of_event(TraceEvent.GOAL)}\n    deadlock_orders = {step.node_order for step in trace.of_event(TraceEvent.DEADLOCK)}\n    visited_orders = {step.node_order for step in trace.of_event(TraceEvent.VISIT)}\n    solution_orders = {node.order for node in result.solution_path}\n    removed = removed_edges(trace)\n\n    lines = [\n        "digraph search {",\n        f\'  label="{_title(result)}"; labelloc=t; fontname="{theme.GRAPH_FONT}";\',\n        f\'  node [shape=box, style="rounded,filled", fontname="{theme.GRAPH_MONO_FONT}", fontsize=10];\',\n        f\'  edge [fontname="{theme.GRAPH_FONT}", fontsize=9];\',\n        _node(root_steps[0], theme.GRAPH_ROOT_FILL, on_path=bool(solution_orders)),\n    ]\n\n    for step in trace.of_event(TraceEvent.GENERATE):\n        if step.parent_order is None or (step.parent_order, step.rule_id) in removed:\n            continue\n        on_path = step.node_order in solution_orders\n        if step.node_order in goal_orders:\n            fill = theme.GRAPH_GOAL_FILL\n        elif step.node_order in deadlock_orders:\n            fill = theme.GRAPH_DEADLOCK_FILL\n        elif step.node_order in visited_orders:\n            fill = theme.GRAPH_VISITED_FILL\n        else:\n            fill = theme.GRAPH_GENERATED_FILL\n        lines.append(_node(step, fill, on_path=on_path))\n        lines.append(_edge(step, on_path=on_path))\n\n    lines.extend(_pruned(step) for step in trace.of_event(TraceEvent.PRUNE))\n    lines.append("}")\n    return "\\n".join(lines)\n\n\ndef _title(result: SearchResult) -> str:\n    parts = [result.problem, result.label, theme.outcome_label(result.outcome)]\n    if result.solution_length is not None:\n        parts.append(f"{result.solution_length} {theme.GRAPH_MOVES_LABEL}")\n    if result.solution_cost is not None:\n        parts.append(f"{theme.GRAPH_COST_LABEL} {result.solution_cost}")\n    return " · ".join(parts)\n\n\ndef _node(step: TraceStep, fill: str, *, on_path: bool) -> str:\n    label = f"#{step.node_order}\\\\n{state_render.render_inline(step.state)}"\n    width = theme.GRAPH_PATH_WIDTH if on_path else 1\n    color = theme.GRAPH_PATH_COLOR if on_path else theme.GRAPH_NODE_COLOR\n    return (\n        f\'  n{step.node_order} [label="{label}", fillcolor="{fill}", \'\n        f\'color="{color}", penwidth={width}];\'\n    )\n\n\ndef _edge(step: TraceStep, *, on_path: bool) -> str:\n    width = theme.GRAPH_PATH_WIDTH if on_path else 1\n    color = theme.GRAPH_PATH_COLOR if on_path else theme.GRAPH_NODE_COLOR\n    return (\n        f\'  n{step.parent_order} -> n{step.node_order} [label="{step.rule_id}", \'\n        f\'color="{color}", fontcolor="{color}", penwidth={width}];\'\n    )\n\n\ndef _pruned(step: TraceStep) -> str:\n    ghost = f"p{step.node_order}_{step.rule_id}"\n    return (\n        f\'  {ghost} [label="{step.rule_id}", shape=plaintext, style=solid, \'\n        f\'fontcolor="{theme.GRAPH_PRUNED_COLOR}", fontsize=8];\\n\'\n        f"  n{step.node_order} -> {ghost} [style=dotted, "\n        f\'color="{theme.GRAPH_PRUNED_COLOR}", arrowhead=none];\'\n    )\n',
    'src/libs/outputs/formatter.py': 'from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Protocol\n\nfrom core.domain.state import State\nfrom core.search_tree.result import SearchResult\nfrom libs.ranking.leaderboard import Leaderboard, Summary\n\n\n@dataclass(frozen=True, slots=True)\nclass ProblemReport:\n    problem_id: str\n    initial_state: State\n    goal_state: State\n    results: tuple[SearchResult, ...]\n    leaderboard: Leaderboard | None = None\n\n\n@dataclass(frozen=True, slots=True)\nclass Report:\n    problems: tuple[ProblemReport, ...]\n    summary: Summary | None = None\n    show_tree: bool = False\n    show_trace: bool = False\n    show_states: bool = False\n    summary_only: bool = False\n\n\nclass Formatter(Protocol):\n    def render(self, report: Report) -> str: ...\n',
    'src/libs/outputs/graph_writer.py': 'from __future__ import annotations\n\nimport shutil\nimport subprocess\nfrom pathlib import Path\n\nfrom libs.outputs import dot_render\nfrom libs.outputs.formatter import Report\n\nDOT_EXECUTABLE = "dot"\n\n\ndef graphviz_available() -> bool:\n    return shutil.which(DOT_EXECUTABLE) is not None\n\n\ndef write_graphs(report: Report, directory: Path) -> tuple[Path, ...]:\n    render = graphviz_available()\n    written: list[Path] = []\n    for entry in report.problems:\n        target = directory / entry.problem_id\n        target.mkdir(parents=True, exist_ok=True)\n        for result in entry.results:\n            source = dot_render.render_dot(result)\n            stem = target / f"{result.algorithm}_{result.strategy}"\n            dot_path = stem.with_suffix(".dot")\n            dot_path.write_text(source, encoding="utf-8")\n            written.append(dot_path)\n            if render:\n                written.append(_render_svg(dot_path, stem.with_suffix(".svg")))\n    return tuple(written)\n\n\ndef _render_svg(dot_path: Path, svg_path: Path) -> Path:\n    subprocess.run(\n        [DOT_EXECUTABLE, "-Tsvg", str(dot_path), "-o", str(svg_path)],\n        check=True,\n        capture_output=True,\n    )\n    return svg_path\n',
    'src/libs/outputs/json_writer.py': 'from __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nfrom core.domain.state import Peg, State\nfrom core.search_tree.result import SearchResult\nfrom libs.outputs import theme\nfrom libs.outputs.formatter import ProblemReport, Report\nfrom libs.ranking.leaderboard import Leaderboard, Stat, Summary, SummaryRow\n\ntype JsonValue = (\n    str | int | float | bool | list["JsonValue"] | dict[str, "JsonValue"] | None\n)\n\n\nclass JsonFormatter:\n    name = "json"\n\n    def render(self, report: Report) -> str:\n        payload: dict[str, JsonValue] = {\n            "problemas": [_problem_to_json(entry) for entry in report.problems]\n        }\n        if report.summary is not None:\n            payload["resumo"] = _summary_to_json(report.summary)\n        return json.dumps(payload, ensure_ascii=False, indent=2)\n\n\ndef write_file(path: Path, text: str) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(text, encoding="utf-8")\n\n\ndef _problem_to_json(entry: ProblemReport) -> JsonValue:\n    payload: dict[str, JsonValue] = {\n        "id": entry.problem_id,\n        "estado_inicial": _state_to_json(entry.initial_state),\n        "objetivo": _state_to_json(entry.goal_state),\n        "execucoes": [_result_to_json(result) for result in entry.results],\n    }\n    if entry.leaderboard is not None:\n        payload["placar"] = _leaderboard_to_json(entry.leaderboard)\n    return payload\n\n\ndef _leaderboard_to_json(leaderboard: Leaderboard) -> JsonValue:\n    return {\n        "modo": leaderboard.mode,\n        "posicoes": [\n            {\n                "posicao": row.position,\n                "algoritmo": row.result.algorithm,\n                "estrategia": row.result.strategy,\n                "desfecho": theme.outcome_label(row.result.outcome),\n                "movimentos": row.result.solution_length,\n                "escore": row.score,\n            }\n            for row in leaderboard.rows\n        ],\n    }\n\n\ndef _summary_to_json(summary: Summary) -> JsonValue:\n    return {\n        "linhas": [\n            {\n                "algoritmo": row.algorithm,\n                "estrategia": row.strategy,\n                "execucoes": row.runs,\n                "sucessos": row.successes,\n                "impasses": row.deadlocks,\n                "movimentos": _stat_to_json(row.moves),\n                "custo": _stat_to_json(row.cost),\n                "iteracoes": _stat_to_json(row.iterations),\n                "nos_gerados": _stat_to_json(row.nodes_generated),\n            }\n            for row in summary.rows\n        ],\n        "melhores": [\n            {\n                "criterio": highlight.criterion,\n                "pela_media": [_row_label(row) for row in highlight.by_mean],\n                "pela_mediana": [_row_label(row) for row in highlight.by_median],\n            }\n            for highlight in summary.highlights\n        ],\n    }\n\n\ndef _row_label(row: SummaryRow) -> JsonValue:\n    return {"algoritmo": row.algorithm, "estrategia": row.strategy}\n\n\ndef _stat_to_json(stat: Stat | None) -> JsonValue:\n    if stat is None:\n        return None\n    return {"media": round(stat.mean, 2), "mediana": stat.median}\n\n\ndef _result_to_json(result: SearchResult) -> JsonValue:\n    metrics = result.metrics\n    return {\n        "algoritmo": result.algorithm,\n        "estrategia": result.strategy,\n        "desfecho": theme.outcome_label(result.outcome),\n        "movimentos": result.solution_length,\n        "custo": result.solution_cost,\n        "caminho": list(result.applied_rules),\n        "metricas": {\n            "iteracoes": metrics.iterations,\n            "nos_gerados": metrics.nodes_generated,\n            "nos_visitados": metrics.nodes_visited,\n            "regras_testadas": metrics.rules_tested,\n            "retrocessos": metrics.backtracks,\n            "impasses": metrics.deadlocks,\n            "profundidade_maxima": metrics.max_depth,\n            "tempo_ms": round(metrics.elapsed_ms, 3),\n        },\n        "estados": [_state_to_json(node.state) for node in result.solution_path],\n    }\n\n\ndef _state_to_json(state: State) -> JsonValue:\n    return {\n        theme.PEG_NAMES[peg]: [theme.disk_name(disk) for disk in state[peg]]\n        for peg in Peg\n    }\n',
    'src/libs/outputs/state_render.py': 'from __future__ import annotations\n\nfrom core.domain.state import CAPACITIES, Peg, State\nfrom libs.outputs import theme\n\n\ndef render_inline(state: State) -> str:\n    parts = []\n    for peg in Peg:\n        stack = state[peg]\n        content = (\n            ",".join(theme.disk_symbol(disk) for disk in stack)\n            if stack\n            else theme.EMPTY_PEG\n        )\n        parts.append(f"{theme.PEG_NAMES[peg]}[{content}]")\n    return " ".join(parts)\n\n\ndef render_pegs(state: State) -> str:\n    height = max(CAPACITIES.values())\n    columns = [_column(state, peg, height) for peg in Peg]\n    width = max(len(cell) for column in columns for cell in column)\n\n    lines = [\n        "  ".join(column[level].center(width) for column in columns).rstrip()\n        for level in range(height)\n    ]\n    footer = "  ".join(\n        f"{theme.PEG_NAMES[peg]}({CAPACITIES[peg]})".center(width) for peg in Peg\n    )\n    lines.append(footer.rstrip())\n    return "\\n".join(lines)\n\n\ndef _column(state: State, peg: Peg, height: int) -> list[str]:\n    stack = state[peg]\n    cells = ["" for _ in range(height - len(stack))]\n    cells.extend(f"[{theme.disk_name(disk)}]" for disk in reversed(stack))\n    capacity = CAPACITIES[peg]\n    for level in range(height):\n        if not cells[level] and height - level <= capacity:\n            cells[level] = "|"\n    return cells\n',
    'src/libs/outputs/theme.py': 'from __future__ import annotations\n\nfrom collections.abc import Mapping\nfrom typing import Final\n\nfrom core.rules.domain.base import Disk, Peg\nfrom core.search_tree.outcome import Outcome\nfrom core.search_tree.trace import TraceEvent\n\nDISK_NAMES: Final[Mapping[Disk, str]] = {\n    Disk.GREEN: "VERDE",\n    Disk.RED: "VERMELHO",\n    Disk.BLUE: "AZUL",\n}\n\nDISK_SYMBOLS: Final[Mapping[Disk, str]] = {\n    Disk.GREEN: "V",\n    Disk.RED: "R",\n    Disk.BLUE: "A",\n}\n\nPEG_NAMES: Final[Mapping[Peg, str]] = {\n    Peg.H1: "H1",\n    Peg.H2: "H2",\n    Peg.H3: "H3",\n}\n\nOUTCOME_LABELS: Final[Mapping[Outcome, str]] = {\n    Outcome.SUCCESS: "SUCESSO",\n    Outcome.DEADLOCK: "IMPASSE",\n    Outcome.CUTOFF: "LIMITE",\n    Outcome.FAILURE: "FALHA",\n}\n\nEVENT_LABELS: Final[Mapping[TraceEvent, str]] = {\n    TraceEvent.ROOT: "raiz",\n    TraceEvent.VISIT: "visita",\n    TraceEvent.GENERATE: "gera",\n    TraceEvent.PRUNE: "poda",\n    TraceEvent.GOAL: "objetivo",\n    TraceEvent.DEADLOCK: "impasse",\n    TraceEvent.BACKTRACK: "retrocesso",\n    TraceEvent.CUTOFF: "limite",\n    TraceEvent.EXHAUSTED: "esgotado",\n}\n\nREPORT_LABELS: Final[Mapping[str, str]] = {\n    "outcome": "Desfecho",\n    "moves": "Movimentos",\n    "cost": "Custo",\n    "path": "Caminho",\n    "iterations": "Iterações",\n    "nodes_generated": "Nós gerados",\n    "nodes_visited": "Nós visitados",\n    "rules_tested": "Regras testadas",\n    "backtracks": "Retrocessos",\n    "deadlocks": "Impasses",\n    "max_depth": "Profundidade máx",\n    "elapsed": "Tempo (ms)",\n}\n\nLEADERBOARD_HEADERS: Final[tuple[str, ...]] = (\n    "#",\n    "ALGORITMO",\n    "ESTRATÉGIA",\n    "DESFECHO",\n    "MOV",\n    "ITER",\n    "GERADOS",\n    "ESCORE",\n)\n\nSUMMARY_HEADERS: Final[tuple[str, ...]] = (\n    "ALGORITMO",\n    "ESTRATÉGIA",\n    "SUCESSOS",\n    "IMPASSES",\n    "MOVIMENTOS",\n    "CUSTO",\n    "ITERAÇÕES",\n    "GERADOS",\n)\n\nSUMMARY_LEGEND: Final = (\n    "Cada métrica: média / mediana. Movimentos e custo contam só os sucessos."\n)\n\nHIGHLIGHTS_HEADERS: Final[tuple[str, ...]] = (\n    "CRITÉRIO",\n    "MELHOR PELA MÉDIA",\n    "MELHOR PELA MEDIANA",\n)\n\nHIGHLIGHTS_LEGEND: Final = (\n    "Menor valor vence. Só concorre quem tem o maior número de sucessos."\n)\n\nALL_STRATEGIES: Final = "(todas)"\n\nCRITERION_LABELS: Final[Mapping[str, str]] = {\n    "moves": "Movimentos",\n    "cost": "Custo",\n    "iterations": "Iterações",\n    "nodes_generated": "Nós gerados",\n}\n\nTREE_HEADER: Final = "Árvore de busca"\nTRACE_HEADER: Final = "Passo a passo"\nSTATES_HEADER: Final = "Estados do caminho solução"\nLEADERBOARD_HEADER: Final = "Placar"\nSUMMARY_HEADER: Final = "Resumo consolidado"\nHIGHLIGHTS_HEADER: Final = "Melhor por critério"\nPROBLEM_HEADER: Final = "PROBLEMA"\nINITIAL_STATE_HEADER: Final = "Posição inicial"\nGOAL_STATE_HEADER: Final = "Objetivo"\n\nABSENT: Final = "—"\nARROW: Final = " → "\nEMPTY_PEG: Final = "·"\n\nBOX_TOP_LEFT: Final = "╭"\nBOX_TOP_RIGHT: Final = "╮"\nBOX_BOTTOM_LEFT: Final = "╰"\nBOX_BOTTOM_RIGHT: Final = "╯"\nBOX_HORIZONTAL: Final = "─"\nBOX_VERTICAL: Final = "│"\n\nTREE_BRANCH: Final = "├── "\nTREE_LAST_BRANCH: Final = "└── "\nTREE_TRUNK: Final = "│   "\nTREE_GAP: Final = "    "\n\nGRAPH_FONT: Final = "Helvetica"\nGRAPH_MONO_FONT: Final = "monospace"\nGRAPH_MOVES_LABEL: Final = "movimentos"\nGRAPH_COST_LABEL: Final = "custo"\nGRAPH_ROOT_FILL: Final = "#dbe9ff"\nGRAPH_GOAL_FILL: Final = "#c6f0c6"\nGRAPH_DEADLOCK_FILL: Final = "#f7c6c6"\nGRAPH_VISITED_FILL: Final = "#ffffff"\nGRAPH_GENERATED_FILL: Final = "#eeeeee"\nGRAPH_NODE_COLOR: Final = "#333333"\nGRAPH_PATH_COLOR: Final = "#1a56b8"\nGRAPH_PRUNED_COLOR: Final = "#999999"\nGRAPH_PATH_WIDTH: Final = 2.5\n\nLABEL_WIDTH: Final = 18\nMIN_BOX_WIDTH: Final = 56\nMAX_BOX_WIDTH: Final = 76\n\n\ndef disk_symbol(disk: Disk) -> str:\n    return DISK_SYMBOLS[disk]\n\n\ndef disk_name(disk: Disk) -> str:\n    return DISK_NAMES[disk]\n\n\ndef outcome_label(outcome: Outcome) -> str:\n    return OUTCOME_LABELS[outcome]\n\n\ndef event_label(event: TraceEvent) -> str:\n    return EVENT_LABELS[event]\n',
    'src/libs/outputs/trace_render.py': 'from __future__ import annotations\n\nfrom core.search_tree.trace import Trace\nfrom libs.outputs import state_render, theme\n\n_HEADERS = ("ITER", "EVENTO", "NÓ", "PROF", "REGRA", "ESTADO")\n\n\ndef render_trace(trace: Trace) -> str:\n    rows = [\n        (\n            str(step.iteration),\n            theme.event_label(step.event),\n            f"#{step.node_order}",\n            str(step.depth),\n            step.rule_id or theme.ABSENT,\n            state_render.render_inline(step.state),\n        )\n        for step in trace\n    ]\n    return render_table(_HEADERS, tuple(rows))\n\n\ndef render_table(headers: tuple[str, ...], rows: tuple[tuple[str, ...], ...]) -> str:\n    widths = [\n        max(len(headers[column]), *(len(row[column]) for row in rows))\n        if rows\n        else len(headers[column])\n        for column in range(len(headers))\n    ]\n    lines = [\n        "  ".join(header.ljust(widths[i]) for i, header in enumerate(headers)).rstrip()\n    ]\n    lines.extend(\n        "  ".join(cell.ljust(widths[i]) for i, cell in enumerate(row)).rstrip()\n        for row in rows\n    )\n    return "\\n".join(lines)\n',
    'src/libs/outputs/tree_render.py': 'from __future__ import annotations\n\nfrom core.search_tree.trace import Trace, TraceEvent\nfrom libs.outputs import state_render, theme\n\n\ndef render_tree(trace: Trace) -> str:\n    root_steps = trace.of_event(TraceEvent.ROOT)\n    if not root_steps:\n        return ""\n\n    root = root_steps[0]\n    removed = removed_edges(trace)\n    children: dict[int, list[tuple[int, str | None, str]]] = {}\n    for step in trace.of_event(TraceEvent.GENERATE):\n        if step.parent_order is None or (step.parent_order, step.rule_id) in removed:\n            continue\n        label = state_render.render_inline(step.state)\n        children.setdefault(step.parent_order, []).append(\n            (step.node_order, step.rule_id, label)\n        )\n\n    goal_orders = {step.node_order for step in trace.of_event(TraceEvent.GOAL)}\n    deadlock_orders = {step.node_order for step in trace.of_event(TraceEvent.DEADLOCK)}\n\n    lines = [f"#{root.node_order} {state_render.render_inline(root.state)}"]\n    _append_children(lines, root.node_order, children, goal_orders, deadlock_orders, "")\n    return "\\n".join(lines)\n\n\ndef _append_children(\n    lines: list[str],\n    parent_order: int,\n    children: dict[int, list[tuple[int, str | None, str]]],\n    goal_orders: set[int],\n    deadlock_orders: set[int],\n    prefix: str,\n) -> None:\n    siblings = children.get(parent_order, ())\n    for position, (order, rule_id, label) in enumerate(siblings):\n        is_last = position == len(siblings) - 1\n        connector = theme.TREE_LAST_BRANCH if is_last else theme.TREE_BRANCH\n        marks = ""\n        if order in goal_orders:\n            marks = f"  ({theme.event_label(TraceEvent.GOAL)})"\n        elif order in deadlock_orders:\n            marks = f"  ({theme.event_label(TraceEvent.DEADLOCK)})"\n        lines.append(f"{prefix}{connector}{rule_id} → #{order} {label}{marks}")\n        _append_children(\n            lines,\n            order,\n            children,\n            goal_orders,\n            deadlock_orders,\n            prefix + (theme.TREE_GAP if is_last else theme.TREE_TRUNK),\n        )\n\n\ndef removed_edges(trace: Trace) -> set[tuple[int, str | None]]:\n    return {\n        (step.node_order, step.rule_id) for step in trace.of_event(TraceEvent.PRUNE)\n    }\n',
    'src/libs/ranking/comparator.py': 'from __future__ import annotations\n\nfrom collections.abc import Iterable\n\nfrom core.search_tree.result import SearchResult\nfrom libs.ranking import criteria\n\n\ndef lexicographic_key(result: SearchResult) -> tuple[int, int, int, int, str]:\n    return (\n        criteria.outcome_rank(result),\n        criteria.solution_length(result),\n        criteria.iterations(result),\n        criteria.nodes_generated(result),\n        criteria.label(result),\n    )\n\n\ndef sort_results(results: Iterable[SearchResult]) -> tuple[SearchResult, ...]:\n    return tuple(sorted(results, key=lexicographic_key))\n',
    'src/libs/ranking/criteria.py': 'from __future__ import annotations\n\nfrom collections.abc import Callable, Mapping\nfrom dataclasses import dataclass\nfrom typing import Final\n\nfrom core.search_tree.outcome import Outcome\nfrom core.search_tree.result import SearchResult\n\nOUTCOME_ORDER: Final[Mapping[Outcome, int]] = {\n    Outcome.SUCCESS: 0,\n    Outcome.DEADLOCK: 1,\n    Outcome.CUTOFF: 2,\n    Outcome.FAILURE: 3,\n}\n\nNO_SOLUTION: Final = 10**6\n\n\ndef outcome_rank(result: SearchResult) -> int:\n    return OUTCOME_ORDER[result.outcome]\n\n\ndef solution_length(result: SearchResult) -> int:\n    length = result.solution_length\n    return NO_SOLUTION if length is None else length\n\n\ndef iterations(result: SearchResult) -> int:\n    return result.metrics.iterations\n\n\ndef nodes_generated(result: SearchResult) -> int:\n    return result.metrics.nodes_generated\n\n\ndef label(result: SearchResult) -> str:\n    return f"{result.algorithm}:{result.strategy}"\n\n\n@dataclass(frozen=True, slots=True)\nclass Criterion:\n    name: str\n    measure: Callable[[SearchResult], int]\n    weight: float\n\n\nCRITERIA: Final[tuple[Criterion, ...]] = (\n    Criterion("outcome", outcome_rank, 0.50),\n    Criterion("solution_length", solution_length, 0.25),\n    Criterion("iterations", iterations, 0.15),\n    Criterion("nodes_generated", nodes_generated, 0.10),\n)\n',
    'src/libs/ranking/leaderboard.py': 'from __future__ import annotations\n\nfrom collections.abc import Callable, Mapping, Sequence\nfrom dataclasses import dataclass\nfrom statistics import fmean, median\n\nfrom core.search_tree.outcome import Outcome\nfrom core.search_tree.result import SearchResult\nfrom libs.ranking import criteria\nfrom libs.ranking.comparator import lexicographic_key, sort_results\nfrom libs.ranking.scorer import score_all\n\n\n@dataclass(frozen=True, slots=True)\nclass LeaderboardRow:\n    position: int\n    result: SearchResult\n    score: float | None\n\n\n@dataclass(frozen=True, slots=True)\nclass Leaderboard:\n    problem_id: str\n    mode: str\n    rows: tuple[LeaderboardRow, ...]\n\n    @property\n    def winner(self) -> LeaderboardRow | None:\n        return self.rows[0] if self.rows else None\n\n\n@dataclass(frozen=True, slots=True)\nclass Stat:\n    mean: float\n    median: float\n\n\n@dataclass(frozen=True, slots=True)\nclass SummaryRow:\n    algorithm: str\n    strategy: str\n    runs: int\n    successes: int\n    deadlocks: int\n    moves: Stat | None\n    cost: Stat | None\n    iterations: Stat\n    nodes_generated: Stat\n\n\n@dataclass(frozen=True, slots=True)\nclass Highlight:\n    criterion: str\n    by_mean: tuple[SummaryRow, ...]\n    by_median: tuple[SummaryRow, ...]\n\n\n@dataclass(frozen=True, slots=True)\nclass Summary:\n    rows: tuple[SummaryRow, ...]\n    highlights: tuple[Highlight, ...] = ()\n\n\nSUMMARY_CRITERIA: tuple[tuple[str, Callable[[SummaryRow], Stat | None]], ...] = (\n    ("moves", lambda row: row.moves),\n    ("cost", lambda row: row.cost),\n    ("iterations", lambda row: row.iterations),\n    ("nodes_generated", lambda row: row.nodes_generated),\n)\n\n\ndef build_leaderboard(\n    problem_id: str, results: Sequence[SearchResult], mode: str\n) -> Leaderboard:\n    scores = score_all(results) if mode == "score" else {}\n    ordered = (\n        _sorted_by_score(results, scores) if mode == "score" else sort_results(results)\n    )\n    rows = tuple(\n        LeaderboardRow(\n            position=position,\n            result=result,\n            score=scores.get(criteria.label(result)),\n        )\n        for position, result in enumerate(ordered, start=1)\n    )\n    return Leaderboard(problem_id=problem_id, mode=mode, rows=rows)\n\n\ndef build_summary(results: Sequence[SearchResult]) -> Summary:\n    grouped: dict[tuple[str, str], list[SearchResult]] = {}\n    for result in results:\n        grouped.setdefault((result.algorithm, result.strategy), []).append(result)\n\n    rows = tuple(\n        _summarise(algorithm, strategy, group)\n        for (algorithm, strategy), group in grouped.items()\n    )\n    return Summary(\n        rows=tuple(sorted(rows, key=_summary_key)), highlights=_highlights(rows)\n    )\n\n\ndef _highlights(rows: Sequence[SummaryRow]) -> tuple[Highlight, ...]:\n    if not rows:\n        return ()\n    most = max(row.successes for row in rows)\n    contenders = [row for row in rows if row.successes == most]\n    highlights: list[Highlight] = []\n    for name, measure in SUMMARY_CRITERIA:\n        measured: list[tuple[SummaryRow, Stat]] = []\n        for row in contenders:\n            stat = measure(row)\n            if stat is not None:\n                measured.append((row, stat))\n        if not measured:\n            continue\n        best_mean = min(stat.mean for _, stat in measured)\n        best_median = min(stat.median for _, stat in measured)\n        highlights.append(\n            Highlight(\n                criterion=name,\n                by_mean=tuple(r for r, s in measured if s.mean == best_mean),\n                by_median=tuple(r for r, s in measured if s.median == best_median),\n            )\n        )\n    return tuple(highlights)\n\n\ndef _stat(values: Sequence[int]) -> Stat | None:\n    if not values:\n        return None\n    return Stat(mean=fmean(values), median=float(median(values)))\n\n\ndef _sorted_by_score(\n    results: Sequence[SearchResult], scores: Mapping[str, float]\n) -> tuple[SearchResult, ...]:\n    return tuple(\n        sorted(\n            results,\n            key=lambda result: (\n                -scores[criteria.label(result)],\n                lexicographic_key(result),\n            ),\n        )\n    )\n\n\ndef _summarise(\n    algorithm: str, strategy: str, group: Sequence[SearchResult]\n) -> SummaryRow:\n    moves = [r.solution_length for r in group if r.solution_length is not None]\n    costs = [r.solution_cost for r in group if r.solution_cost is not None]\n    iterations = _stat([result.metrics.iterations for result in group])\n    generated = _stat([result.metrics.nodes_generated for result in group])\n    assert iterations is not None and generated is not None\n    return SummaryRow(\n        algorithm=algorithm,\n        strategy=strategy,\n        runs=len(group),\n        successes=sum(result.outcome.is_success for result in group),\n        deadlocks=sum(result.outcome is Outcome.DEADLOCK for result in group),\n        moves=_stat(moves),\n        cost=_stat(costs),\n        iterations=iterations,\n        nodes_generated=generated,\n    )\n\n\ndef _summary_key(row: SummaryRow) -> tuple[int, float, float, float, str, str]:\n    no_solution = float(criteria.NO_SOLUTION)\n    return (\n        -row.successes,\n        row.moves.mean if row.moves is not None else no_solution,\n        row.moves.median if row.moves is not None else no_solution,\n        row.iterations.mean,\n        row.algorithm,\n        row.strategy,\n    )\n',
    'src/libs/ranking/scorer.py': 'from __future__ import annotations\n\nfrom collections.abc import Mapping, Sequence\n\nfrom core.search_tree.result import SearchResult\nfrom libs.ranking import criteria\nfrom libs.ranking.criteria import Criterion\n\nMAX_SCORE = 100.0\n\n\ndef score_all(results: Sequence[SearchResult]) -> Mapping[str, float]:\n    if not results:\n        return {}\n    best = {\n        criterion.name: min(criterion.measure(result) for result in results)\n        for criterion in criteria.CRITERIA\n    }\n    return {criteria.label(result): _score(result, best) for result in results}\n\n\ndef _score(result: SearchResult, best: Mapping[str, int]) -> float:\n    total = sum(\n        criterion.weight * _normalise(criterion, result, best[criterion.name])\n        for criterion in criteria.CRITERIA\n    )\n    return round(total * MAX_SCORE, 2)\n\n\ndef _normalise(criterion: Criterion, result: SearchResult, best: int) -> float:\n    value = criterion.measure(result)\n    if value == best:\n        return 1.0\n    if value <= 0:\n        return 1.0\n    return best / value if best > 0 else 1.0 / (1.0 + value)\n',
    'src/runner/cli.py': 'from __future__ import annotations\n\nfrom collections.abc import Sequence\nfrom pathlib import Path\n\nfrom config import settings\nfrom config.logging import configure_logging\nfrom core.algorithms.domain.registry import ALGORITHM_NAMES\nfrom core.domain.problem import PROBLEMS\nfrom core.rules.domain.exceptions import InvalidRuleOrderError\nfrom core.rules.strategies.domain.registry import STRATEGY_NAMES\nfrom libs.inputs import parser, validator\nfrom libs.inputs.exceptions import (\n    InputError,\n    InvalidArgumentError,\n    UnknownSelectionError,\n)\nfrom libs.inputs.selection import ExecutionRequest\nfrom libs.outputs import console, graph_writer, state_render\nfrom libs.outputs.console import ConsoleFormatter\nfrom libs.outputs.formatter import Formatter, Report\nfrom libs.outputs.json_writer import JsonFormatter, write_file\nfrom runner.pipeline import build_report\n\n_FORMATTERS: dict[str, Formatter] = {\n    ConsoleFormatter.name: ConsoleFormatter(),\n    JsonFormatter.name: JsonFormatter(),\n}\n\n\ndef main(argv: Sequence[str] | None = None) -> int:\n    arguments = [] if argv is None else list(argv)\n    try:\n        request = validator.validate(parser.parse(arguments))\n    except (InputError, InvalidRuleOrderError) as error:\n        console.write_error(_error_message(error))\n        console.write_error(settings.ERROR_HINT)\n        return settings.EXIT_INVALID_INPUT\n\n    configure_logging(verbose=request.verbose)\n\n    if request.show_help:\n        console.write(parser.help_text())\n        return settings.EXIT_OK\n\n    if request.list_only:\n        console.write(_render_catalogues())\n        return settings.EXIT_OK\n\n    report = build_report(request)\n    _emit(report, request)\n    if request.svg_dir is not None:\n        _write_graphs(report, request.svg_dir)\n    return settings.EXIT_OK\n\n\ndef _write_graphs(report: Report, directory: Path) -> None:\n    written = graph_writer.write_graphs(report, directory)\n    console.write(settings.GRAPHS_WRITTEN.format(count=len(written), path=directory))\n    if not graph_writer.graphviz_available():\n        console.write_error(settings.GRAPHVIZ_MISSING)\n\n\ndef _emit(report: Report, request: ExecutionRequest) -> None:\n    text = _FORMATTERS[request.output_format].render(report)\n    if request.output_path is None:\n        console.write(text)\n        return\n    write_file(request.output_path, text)\n    console.write(settings.OUTPUT_WRITTEN.format(path=request.output_path))\n\n\ndef _render_catalogues() -> str:\n    problems = "\\n".join(\n        f"  {problem.id}  {state_render.render_inline(problem.goal)}"\n        for problem in PROBLEMS\n    )\n    return "\\n\\n".join(\n        [\n            console.render_section(settings.LIST_PROBLEMS_HEADER, problems),\n            console.render_section(\n                settings.LIST_ALGORITHMS_HEADER,\n                "\\n".join(f"  {name}" for name in ALGORITHM_NAMES),\n            ),\n            console.render_section(\n                settings.LIST_STRATEGIES_HEADER,\n                "\\n".join(f"  {name}" for name in STRATEGY_NAMES),\n            ),\n        ]\n    )\n\n\ndef _error_message(error: Exception) -> str:\n    match error:\n        case UnknownSelectionError():\n            detail = settings.ERROR_UNKNOWN_SELECTION.format(\n                kind=settings.KIND_LABELS[error.kind],\n                name=error.name,\n                available=", ".join(error.available),\n            )\n        case InvalidRuleOrderError():\n            detail = settings.ERROR_INVALID_RULE_ORDER.format(reason=error.reason)\n        case InvalidArgumentError():\n            detail = settings.ERROR_INVALID_ARGUMENT.format(detail=error.detail)\n        case _:\n            detail = str(error)\n    return f"{settings.ERROR_PREFIX} {detail}"\n',
    'src/runner/executor.py': 'from __future__ import annotations\n\nfrom core.algorithms.domain.registry import get_algorithm\nfrom core.domain.problem import Problem, all_goal_problems, get_problem\nfrom core.rules.strategies.custom_order import CUSTOM, CustomOrderStrategy\nfrom core.rules.strategies.domain.base import ControlStrategy\nfrom core.rules.strategies.domain.registry import get_strategy\nfrom core.search_tree.result import SearchResult\nfrom core.search_tree.tree import SearchTree\nfrom libs.inputs.selection import ExecutionRequest\n\n\ndef run_problem(\n    problem: Problem, request: ExecutionRequest\n) -> tuple[SearchResult, ...]:\n    return tuple(\n        get_algorithm(algorithm_name)(\n            SearchTree(),\n            resolve_strategy(strategy_name, request.custom_order),\n            max_iterations=request.max_iterations,\n        ).solve(problem)\n        for algorithm_name in request.algorithm_names\n        for strategy_name in request.strategy_names\n    )\n\n\ndef run_matrix(\n    request: ExecutionRequest,\n) -> tuple[tuple[Problem, tuple[SearchResult, ...]], ...]:\n    problems = (\n        all_goal_problems()\n        if request.all_goals\n        else tuple(get_problem(problem_id) for problem_id in request.problem_ids)\n    )\n    return tuple((problem, run_problem(problem, request)) for problem in problems)\n\n\ndef resolve_strategy(\n    name: str, custom_order: tuple[str, ...] | None\n) -> ControlStrategy:\n    if name == CUSTOM.name and custom_order is not None:\n        return CustomOrderStrategy(sequence=custom_order)\n    return get_strategy(name)\n',
    'src/runner/pipeline.py': 'from __future__ import annotations\n\nfrom libs.inputs.selection import ExecutionRequest\nfrom libs.outputs.formatter import ProblemReport, Report\nfrom libs.ranking.leaderboard import build_leaderboard, build_summary\nfrom runner.executor import run_matrix\n\n\ndef build_report(request: ExecutionRequest) -> Report:\n    executed = run_matrix(request)\n    problems = tuple(\n        ProblemReport(\n            problem_id=problem.id,\n            initial_state=problem.initial,\n            goal_state=problem.goal,\n            results=results,\n            leaderboard=build_leaderboard(problem.id, results, request.rank_mode),\n        )\n        for problem, results in executed\n    )\n    every_result = [result for _, results in executed for result in results]\n    return Report(\n        problems=problems,\n        summary=build_summary(every_result),\n        show_tree=request.show_tree,\n        show_trace=request.show_trace,\n        show_states=request.show_states,\n        summary_only=request.all_goals,\n    )\n',
    'main.py': 'from __future__ import annotations\n\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent / "src"))\n\nfrom runner.cli import main\n\nif __name__ == "__main__":\n    sys.exit(main(sys.argv[1:]))\n',
}

for name, text in FILES.items():
    target = Path(name)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(text, encoding="utf-8")

if str(Path("src").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("src").resolve()))
print(f"{len(FILES)} arquivos gravados em {Path.cwd()}")

In [ ]:
#@title Funções de apresentação
import shutil
import subprocess

import pandas as pd
from IPython.display import SVG, display

from libs.outputs import dot_render, state_render, theme, trace_render, tree_render

LABELS = theme.REPORT_LABELS


def metrics_table(results):
    rows = []
    for result in results:
        metrics = result.metrics
        rows.append(
            {
                "Estratégia": result.strategy,
                LABELS["outcome"]: theme.outcome_label(result.outcome),
                LABELS["moves"]: result.solution_length,
                LABELS["cost"]: result.solution_cost,
                LABELS["path"]: " → ".join(result.applied_rules) or theme.ABSENT,
                LABELS["iterations"]: metrics.iterations,
                LABELS["nodes_generated"]: metrics.nodes_generated,
                LABELS["nodes_visited"]: metrics.nodes_visited,
                LABELS["backtracks"]: metrics.backtracks,
                LABELS["deadlocks"]: metrics.deadlocks,
                LABELS["max_depth"]: metrics.max_depth,
            }
        )
    return (
        pd.DataFrame(rows)
        .set_index("Estratégia")
        .astype({LABELS["moves"]: "Int64", LABELS["cost"]: "Int64"})
    )


def show_path(result):
    for step, node in enumerate(result.solution_path):
        rule = "início" if node.rule is None else f"{node.rule.id}: {node.rule.origin.name} → {node.rule.destination.name}"
        print(f"passo {step}  {rule}  (custo acumulado {node.cost})")
        print(state_render.render_pegs(node.state))
        print()


def show_tree(result):
    source = dot_render.render_dot(result)
    if shutil.which("dot") is None:
        print(tree_render.render_tree(result.trace))
        return
    svg = subprocess.run(
        ["dot", "-Tsvg"], input=source, capture_output=True, text=True, check=True
    ).stdout
    display(SVG(svg))


def show_trace(result):
    print(trace_render.render_trace(result.trace))

## Fluxograma

In [ ]:
#@title Desenha o fluxograma
FLOW = '''digraph flow {
  rankdir=TB; fontname="Helvetica"; bgcolor="transparent";
  node [fontname="Helvetica", fontsize=11, shape=box, style="rounded,filled", fillcolor="#ffffff", color="#333333"];
  edge [fontname="Helvetica", fontsize=10, color="#333333"];
  root [label="raiz na fila", shape=oval, fillcolor="#dbe9ff"];
  pop [label="tira o primeiro da fila"];
  goal [label="é o objetivo?", shape=diamond];
  success [label="SUCESSO", shape=oval, fillcolor="#c6f0c6"];
  visit [label="visita: regras aplicáveis na ordem da estratégia,\nmenos as que levam a um estado já gerado por qualquer ramo"];
  apply [label="aplica todas e coloca os filhos no fim da fila"];
  root -> pop;
  pop -> goal;
  goal -> success [label="sim"];
  goal -> visit [label="não"];
  visit -> apply;
  apply -> pop;
}'''
if shutil.which("dot"):
    display(SVG(subprocess.run(["dot", "-Tsvg"], input=FLOW, capture_output=True, text=True, check=True).stdout))
else:
    print(FLOW)

## O código

A fronteira é uma `QueueFrontier`. A poda é global: `_is_repetition` é sobrescrita para descartar qualquer estado que já esteja em ABERTOS ou FECHADOS, não só os do caminho. O docstring traz a expansão de P1 nível por nível.

In [ ]:
%%writefile src/core/algorithms/breadth_first.py
"""BUSCA EM LARGURA

1. Estado inicial e objetivo (carta P1)

    S0 = ([V, R], [A], [])          Sf = ([], [R, V], [A])

2. Critério

    As regras são analisadas na ordem crescente R1 → R2 → R3 → R4 → R5 → R6.
    Em vez de escolher uma, a busca aplica todas as válidas e coloca os
    filhos no fim de ABERTOS. O estado visitado vai para FECHADOS. Uma
    regra cujo resultado já está em ABERTOS ou FECHADOS é descartada (poda).

3. Execução

    Nível 0
        ABERTOS = [S0]               FECHADOS = []

    Expansão de S0 = ([V, R], [A], [])
        R1 → S1 = ([V], [A, R], [])
        R2 → S2 = ([V], [A], [R])
        R3 → S3 = ([V, R, A], [], [])
        R4 → S4 = ([V, R], [], [A])
        R5, R6: inválidas, H3 vazia
        ABERTOS = [S1, S2, S3, S4]   FECHADOS = [S0]

    Nível 1

    Expansão de S1 = ([V], [A, R], [])
        R1: inválida, H2 cheia
        R2 → S5 = ([], [A, R], [V])
        R3 → S0: poda
        R4 → S2: poda
        ABERTOS = [S2, S3, S4, S5]   FECHADOS = [S0, S1]

    Expansão de S2 = ([V], [A], [R])
        R1 → S6 = ([], [A, V], [R])
        R3 → S7 = ([V, A], [], [R])
        R5 → S0, R6 → S1: poda
        ABERTOS = [S3, S4, S5, S6, S7]

    Expansão de S3 = ([V, R, A], [], [])
        R1 → S0, R2 → S4: poda.  Nada novo.

    Expansão de S4 = ([V, R], [], [A])
        R1 → S8 = ([V], [R], [A])
        R5 → S3, R6 → S0: poda
        ABERTOS = [S5, S6, S7, S8]   FECHADOS = [S0, S1, S2, S3, S4]

    Nível 2
        S5 gera S9.  S6 gera S10.  S7 gera S11 e S12.
        S8 = ([V], [R], [A]) gera:
            R1 → S13 = ([], [R, V], [A])   ← é Sf, entra na fila
            R6 → S14 = ([V], [R, A], [])
        ABERTOS = [S9, S10, S11, S12, S13, S14]

    Nível 3
        S9 gera S15 e S16.  S10, S11 e S12 só geram repetidos.
        Chega a vez de S13 = ([], [R, V], [A])
        ✓ SUCESSO

4. Árvore de busca

    S0 ([V,R], [A], [])
    ├── R1 → S1 ([V], [A,R], [])
    │   └── R2 → S5 ([], [A,R], [V])
    │       └── R3 → S9 ([R], [A], [V])
    │           ├── R3 → S15
    │           └── R5 → S16
    ├── R2 → S2 ([V], [A], [R])
    │   ├── R1 → S6 ([], [A,V], [R])
    │   │   └── R5 → S10
    │   └── R3 → S7 ([V,A], [], [R])
    │       ├── R5 → S11
    │       └── R6 → S12
    ├── R3 → S3 ([V,R,A], [], [])
    └── R4 → S4 ([V,R], [], [A])
        └── R1 → S8 ([V], [R], [A])
            ├── R1 → S13 ([], [R,V], [A])   (objetivo)
            └── R6 → S14 ([V], [R,A], [])

5. Caminho solução

    S0 ([V,R], [A], [])
     │ R4: azul H2 → H3
     ↓
    S4 ([V,R], [], [A])
     │ R1: vermelho H1 → H2
     ↓
    S8 ([V], [R], [A])
     │ R1: verde H1 → H2
     ↓
    Sf ([], [R,V], [A])

    Número de movimentos: 3      Iterações: 14      Gerados: 17

6. Conclusão

    Nada de profundidade d+1 é visitado antes de esgotar a profundidade d,
    então o primeiro caminho até um estado é o mais curto até ele. O objetivo
    encontrado é o ótimo em movimentos, e por isso este método serve de
    referência de comprimento para os outros. A estratégia só muda a ordem
    dentro de um nível.

    A poda aqui é global: um estado descoberto por qualquer ramo nunca é
    gerado de novo. A irrevogável e o backtracking só evitam repetir dentro
    do próprio caminho.
"""

from __future__ import annotations

from typing import ClassVar

from config import settings
from core.algorithms.domain.base import SearchAlgorithm
from core.domain.problem import Problem
from core.domain.state import State
from core.rules.domain.base import TransitionRule
from core.rules.strategies.domain.base import ControlStrategy
from core.search_tree.frontier import QueueFrontier
from core.search_tree.node import Node
from core.search_tree.outcome import Outcome
from core.search_tree.tree import SearchTree


class BreadthFirstSearch(SearchAlgorithm):
    name: ClassVar[str] = "breadth_first"

    def __init__(
        self,
        tree: SearchTree,
        strategy: ControlStrategy,
        *,
        max_iterations: int = settings.MAX_ITERATIONS,
    ) -> None:
        super().__init__(tree, strategy, max_iterations=max_iterations)
        self._closed: set[State] = set()
        self._open_states: set[State] = set()

    def _search(self, problem: Problem) -> Outcome:
        context = self._require_context()
        frontier = QueueFrontier()
        frontier.push(context.root)

        self._closed = set()
        self._open_states = {context.root.state}

        while len(frontier):
            if not self._next_iteration():
                return Outcome.CUTOFF

            node = frontier.pop()
            self._open_states.discard(node.state)
            if problem.is_goal(node.state):
                return self._succeed(node)

            self._closed.add(node.state)
            self._visit(node)
            for rule in self._applicable_rules(node):
                child = self._expand(node, rule)
                self._open_states.add(child.state)
                frontier.push(child)

        return self._exhausted()

    def _is_repetition(
        self, node: Node, rule: TransitionRule, successor: State
    ) -> bool:
        return successor in self._closed or successor in self._open_states

## Execução em P1

Uma execução por estratégia de controle, sobre a carta P1. O motor conta as métricas; a tabela só as organiza.

In [ ]:
from core.algorithms.domain.registry import get_algorithm
from core.domain.problem import get_problem
from core.rules.strategies.domain.registry import STRATEGIES
from core.search_tree.tree import SearchTree

problem = get_problem("P1")
Algorithm = get_algorithm("breadth_first")
results = {
    name: Algorithm(SearchTree(), strategy).solve(problem)
    for name, strategy in STRATEGIES.items()
}
metrics_table(results.values())

## Passo a passo

O trace da execução com a estratégia `ascending`: cada evento do motor (visita, gera, poda, impasse, retrocesso, objetivo), com a iteração, o nó, a profundidade, a regra e o estado. Legenda dos discos: V verde, R vermelho, A azul.

In [ ]:
show_trace(results["ascending"])

## Árvores de busca

Uma árvore por estratégia. O caminho solução está em azul, o objetivo em verde, os impasses em vermelho, os nós gerados e não visitados em cinza e as regras podadas em pontilhado.

In [ ]:
for result in results.values():
    show_tree(result)

## Caminho solução

O caminho da execução com a estratégia `ascending`, estado por estado.

In [ ]:
show_path(results["ascending"])

## Experimente

Troque a ordem das regras ou a carta e rode de novo. `all_goal_problems()` traz os 36 objetivos possíveis (G01 a G36).

In [ ]:
from core.domain.problem import all_goal_problems
from core.rules.strategies.custom_order import CustomOrderStrategy

order = ("R1", "R4", "R2", "R6", "R3", "R5")
target = all_goal_problems()[20]
result = Algorithm(SearchTree(), CustomOrderStrategy(sequence=order)).solve(target)
print(target.id, state_render.render_inline(target.goal))
print(theme.outcome_label(result.outcome), " → ".join(result.applied_rules) or theme.ABSENT)
show_tree(result)

## Conclusão

- Em P1 as três estratégias acham R4, R1, R1: 3 movimentos, o ótimo. A estratégia só muda a ordem dentro de um nível e, com ela, quantas iterações a busca leva.
- O preço é a memória: a fila guarda o nível inteiro, e a busca gera mais nós que a irrevogável e o backtracking com `ascending`.
- Por achar o caminho mais curto, este método é a referência de comprimento para os outros. Em custo, a referência é a busca ordenada.